# Experiment H4: a stronger non-neural baseline (H3 calendar features + mean reversion) (Phase 4)

**Pre-registered 2026-10-19** in [`PHASE4_NEURAL_EVIDENCE_REVIEW`](../docs/research/PHASE4_NEURAL_EVIDENCE_REVIEW.md) section 3 (table row H4). Generated by `scripts/build_colab_notebook.py` from the unit-tested package; **not yet executed**: run it once on Kaggle (GPU).

**Question.** The only thing that has cleared the pre-registered rule so far is a *baseline* (PCA-ridge with the H3 calendar block). Before any neural model is judged, the bar it must beat should be the strongest honest baseline.

**What changes (one thing only vs H3).** Ridge, PCA-ridge, rolling-mean and persistence get the H3 calendar block **plus a mean-reversion block**: for every cell, the filled surface at the origin minus its trailing 20-row mean (for the PCA input: the same on the 3 factor scores). Causal (rows <= t), unit-tested. Ridge penalty grid = H1 grid, validation = previous year (as E-2/H3). **No neural model is trained.** Per the owner's 2026-10-19 instruction the notebook still runs on a **GPU runtime** with CUDA verified and logged; the ridge/PCA/persistence arithmetic stays NumPy float64 (porting it to CUDA would change the numbers and is not done).

**Success / null.** Descriptive only. The result sets the baseline that H5/H6 neural models must beat; if the mean-reversion block changes PCA-ridge skill by less than 0.01 (absolute) under both labels versus H3, the H3 baseline stands.

**Setting (identical to E-2 except the one change above).** Hosted SPY surfaces (Kaggle `dudesurfin/spy-options-eod-volatility-surface-2010-2023`, upstream licence **unverified**); walk-forward test years 2016-2023; models persistence, rolling mean, ridge, PCA-ridge, MLP, GRU (3 seeds + seed-average, fixed hyper-parameters, no search); both labels (E-1 nearest-expiry, E-2 fixed-expiry). Ridge/PCA-ridge use the **H1 penalty grid** (up to 1e6; H1b showed a wider grid changes nothing), so the baselines are the ones that stand after H1/H1b. **Neural training is GPU-only** (no CPU fallback): choose *Accelerator: GPU* in Kaggle.

**Pre-registered decision rule (descriptive; owner-approved 2026-10-16).** A model "improves on persistence" only if (a) pooled MSE skill > 0 **and** pooled MAE below persistence, (b) better by daily MAE in >= 6 of 8 test years, (c) every seed's pooled skill > 0, (d) skill > 0 in >= 2 of 3 realised-vol regimes with n >= 100 cells, (e) under **both** labels; otherwise "holds under one label only" or "no improvement shown". No p-values, no intervals. **A neural result worse than persistence is reported as such and retains the NO-GO.** Test years are evaluated once per frozen configuration (GR-LEAK-05): nothing is selected on them; the configuration hash is printed and saved. No test-window access log exists (known gap).

**How to run on Kaggle.** New notebook -> File -> Import notebook (upload this `.ipynb`) -> *+ Add Input* -> dataset `dudesurfin/spy-options-eod-volatility-surface-2010-2023` -> Accelerator **GPU (T4 or P100)** -> *Save & Run All (Commit)*. Then copy the final "RESULT SUMMARY" line and `report.md` back. Runtime is of the order of E-2 (the GPU phase dominates).


## 2. Configuration
Everything adjustable lives in the next cell. `RUN_MODE='kaggle'` is the real-data run; `'synthetic_plumbing'` uses a labelled SYNTHETIC chain only to check the pipeline and GPU end to end (its numbers say nothing about markets).

In [ ]:
# ===== CONFIGURATION (the only cell you normally edit) =====
RUN_MODE = "kaggle"            # "kaggle" (real hosted data) | "synthetic_plumbing" (labelled SYNTHETIC)
KAGGLE_DATASET = "dudesurfin/spy-options-eod-volatility-surface-2010-2023"
MAX_ROWS = None                # int caps rows read (smoke tests); None = the full dataset
SYNTH_DAYS = 1300              # synthetic_plumbing only: number of business days generated

# quote / expiry filters
MIN_DTE, MAX_DTE = 2, 400
MIN_MID, MAX_REL_SPREAD = 0.05, 0.35
MAX_ABS_K = 0.30               # |ln(K/F)| kept

# surface grid (moneyness k = ln(K/F), maturities in calendar days)
TENOR_DAYS = (7, 14, 30, 60, 91, 182)
K_NODES = (-0.20, -0.12, -0.08, -0.04, -0.02, 0.0, 0.02, 0.04, 0.08)

# forecasting problem
HORIZON = 1                    # trading days ahead
SEQ_LEN = 20                   # GRU look-back (rows)
FIRST_TEST_YEAR, LAST_TEST_YEAR, MIN_TRAIN_YEARS = 2016, 2023, 3   # expanding-window walk-forward

# neural training (fixed a priori; there is no hyper-parameter search)
MODELS = ("mlp", "gru")
SEEDS = (0, 1, 2)
MAX_EPOCHS, PATIENCE, BATCH_SIZE, LR, WEIGHT_DECAY = 150, 15, 128, 1e-3, 1e-4

# outputs
USE_DRIVE = False              # True mounts Google Drive (asks for your consent) so a run survives a runtime reset
DRIVE_DIR = "/content/drive/MyDrive/nvs_runs"
LOCAL_DIR = "/content/nvs_runs"

# ---- PRE-REGISTERED H4 SETTINGS (2026-10-19): DO NOT EDIT ----
VARIANT = "H4"
VAL_MODE = "prev_year"        # "prev_year" (E-1/E-2) | "tail_frac" (H2)
VAL_FRAC = 0.2                 # only used by "tail_frac"
CALENDAR = True          # True = H3 calendar block (days-to-expiry per tenor + weekday one-hot)
MR_WINDOW = 20               # None = off; else trailing-mean window for the mean-reversion block (H4/H5)
PCA_FACTORS = None            # None = cell-level networks; else networks predict this many PCA-factor changes (H5)
ALPHAS_H1 = (0.1, 1.0, 10.0, 100.0, 1000.0, 3000.0, 1.0e4, 3.0e4, 1.0e5, 3.0e5, 1.0e6)   # baselines that stand after H1/H1b
MIN_YEARS_BETTER, MIN_REGIMES, MIN_CELLS_REGIME = 6, 2, 100


## 3. GPU verification
Neural work needs a real CUDA GPU. There is **no CPU fallback**: if no GPU is found, the neural cells stop with instructions.

In [ ]:
import subprocess
try:
    import torch
    GPU_OK = bool(torch.cuda.is_available())
    TORCH_VERSION = torch.__version__
except Exception as exc:                       # torch missing or unusable
    torch, GPU_OK, TORCH_VERSION = None, False, f"import failed: {exc!r}"
print("PyTorch:", TORCH_VERSION)
if GPU_OK:
    prop = torch.cuda.get_device_properties(0)
    print(f"CUDA GPU: {prop.name} | {prop.total_memory / 2**30:.1f} GiB | compute capability {prop.major}.{prop.minor} | CUDA {torch.version.cuda}")
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("NO CUDA GPU is available. Data, surface and baseline sections can still run, but every neural cell will refuse to run.\n"
          "Fix: Runtime -> Change runtime type -> Hardware accelerator: GPU (T4 or better) -> Save, then Runtime -> Restart session and run again from the top.")


## 4. Environment setup
Only `kagglehub` is installed (pinned); NumPy, pandas, SciPy, PyArrow, Matplotlib and PyTorch come with Colab. The package source follows as `%%writefile` cells (generated from `backend/src/volatility_lab`; do not edit here).

In [ ]:
import importlib.util, os, subprocess, sys
# Kaggle Notebook: the dataset is attached natively, no client needed. Colab can also create an EMPTY /kaggle/input (dataset side panel), so require a Kaggle kernel env var or a non-Colab runtime.
ON_KAGGLE = os.path.exists("/kaggle/input") and (bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or importlib.util.find_spec("google.colab") is None)
if not ON_KAGGLE and importlib.util.find_spec("kagglehub") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub==0.4.3"], check=True)
print("environment:", "Kaggle Notebook" if ON_KAGGLE else "Colab / other", "| kagglehub", "not needed" if ON_KAGGLE else "ready")


In [ ]:
import os
for d in ['volatility_lab', 'volatility_lab/api', 'volatility_lab/config', 'volatility_lab/datasets', 'volatility_lab/evaluation', 'volatility_lab/experiments', 'volatility_lab/features', 'volatility_lab/ingestion', 'volatility_lab/models', 'volatility_lab/pricing', 'volatility_lab/storage', 'volatility_lab/surfaces', 'volatility_lab/validation']:
    os.makedirs(f'/content/nvs/src/{d}', exist_ok=True)
print('package directories ready')

In [ ]:
%%writefile /content/nvs/src/volatility_lab/__init__.py
"""volatility_lab"""


In [ ]:
%%writefile /content/nvs/src/volatility_lab/api/__init__.py



In [ ]:
%%writefile /content/nvs/src/volatility_lab/api/server.py
"""Standard-library HTTP transport for the read-only research API (Phase 5). Local use only.

* GET only (every other method -> 405); loopback bind enforced because no authentication exists yet (Phase 6).
* ``dispatch`` is a pure function (path -> status, content type, body) so routes are testable without sockets.
* All logic lives in ``service.RunStore``; this module only maps URLs to it and errors to status codes.

    python -m volatility_lab.api.server --root <runs dir> [--port 8765]
"""
from __future__ import annotations

import argparse
import json
import re
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit

from volatility_lab.api.service import BadRequest, NotFound, RunStore

FRONTEND = Path(__file__).resolve().parents[4] / "frontend" / "index.html"
LOOPBACK = ("127.0.0.1", "localhost", "::1")
HEADERS = {
    "X-Content-Type-Options": "nosniff", "Cache-Control": "no-store", "Referrer-Policy": "no-referrer",
    "Content-Security-Policy": "default-src 'none'; script-src 'unsafe-inline'; style-src 'unsafe-inline'; "
                               "connect-src 'self'; base-uri 'none'; form-action 'none'",
}
_SURFACE = re.compile(r"^/api/runs/([^/]+)/surface/(\d{1,7})$")
_FORECAST = re.compile(r"^/api/runs/([^/]+)/forecasts/([^/]+)/(\d{1,7})$")
_TABLE = re.compile(r"^/api/runs/([^/]+)/tables/([^/]+)$")
_RUN = re.compile(r"^/api/runs/([^/]+)$")


def _json(code: int, obj) -> tuple:
    return code, "application/json", json.dumps(obj, allow_nan=False).encode()


def dispatch(store: RunStore, raw_path: str, frontend: Path = FRONTEND) -> tuple:
    path = urlsplit(raw_path).path
    try:
        if path in ("/", "/index.html"):
            return 200, "text/html; charset=utf-8", frontend.read_bytes()
        if path == "/api/runs":
            return _json(200, store.list_runs())
        if m := _SURFACE.match(path):
            return _json(200, store.surface(m.group(1), int(m.group(2))))
        if m := _FORECAST.match(path):
            return _json(200, store.forecast(m.group(1), m.group(2), int(m.group(3))))
        if m := _TABLE.match(path):
            return _json(200, store.table(m.group(1), m.group(2)))
        if m := _RUN.match(path):
            return _json(200, store.run_info(m.group(1)))
        return _json(404, {"error": "not found"})
    except BadRequest as e:
        return _json(400, {"error": str(e)})
    except NotFound as e:
        return _json(404, {"error": str(e)})
    except FileNotFoundError:
        return _json(404, {"error": "not found"})


def make_handler(store: RunStore, frontend: Path = FRONTEND):
    class Handler(BaseHTTPRequestHandler):
        def _send(self, code: int, ctype: str, body: bytes, extra: dict) -> None:
            self.send_response(code)
            self.send_header("Content-Type", ctype)
            self.send_header("Content-Length", str(len(body)))
            for k, v in {**HEADERS, **extra}.items():
                self.send_header(k, v)
            self.end_headers()
            self.wfile.write(body)

        def do_GET(self):  # noqa: N802
            self._send(*dispatch(store, self.path, frontend), {})

        def _refuse(self):
            self._send(*_json(405, {"error": "read-only API"}), {"Allow": "GET"})

        do_POST = do_PUT = do_DELETE = do_PATCH = _refuse  # noqa: N815

        def log_message(self, *_):  # no request logging
            pass

    return Handler


def create_server(root, host: str = "127.0.0.1", port: int = 8765, frontend: Path = FRONTEND) -> ThreadingHTTPServer:
    if host not in LOOPBACK:
        raise ValueError("refusing non-loopback bind: the API has no authentication (Phase 6 scope)")
    return ThreadingHTTPServer((host, port), make_handler(RunStore(root), frontend))


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--root", required=True, help="directory containing one sub-directory per run")
    ap.add_argument("--port", type=int, default=8765)
    a = ap.parse_args()
    srv = create_server(a.root, port=a.port)
    print(f"serving {a.root} on http://127.0.0.1:{a.port}/ (read-only, local)")
    try:
        srv.serve_forever()
    except KeyboardInterrupt:
        pass


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/nvs/src/volatility_lab/api/service.py
"""Read-only research-API logic (Phase 5). No HTTP here; ``server.py`` is the transport.

A *runs root* holds one directory per run::

    <run_id>/panel.npz        surface panel arrays (``build_surface_panel`` keys)      required
    <run_id>/meta.json        {"origin": "synthetic"|"real", "description": "..."}      optional
    <run_id>/manifest.json    ``dataset-1`` manifest from ``storage.raw_store.ingest``  optional
    <run_id>/tables/*.csv     descriptive result tables                                 optional
    <run_id>/forecasts/*.npz  stored forecasts: target_index, origin_index, pred (n,T,K)  optional

This layer never computes IV, never trains, never forecasts and never holds provider credentials: it only
reads files that already exist and says what they are. A run without ``meta.json`` is reported as
``unlabelled`` so the viewer cannot mistake it for labelled real or synthetic data.
"""
from __future__ import annotations

import csv
import json
import re
from pathlib import Path

import numpy as np

from volatility_lab.storage.raw_store import panel_sha256
from volatility_lab.surfaces.grid import OBSERVED, STATUS_LABELS

NAME = re.compile(r"^[A-Za-z0-9][A-Za-z0-9_.-]{0,79}$")
MAX_TABLE_ROWS = 5000
ORIGINS = ("synthetic", "real")


class NotFound(LookupError):
    pass


class BadRequest(ValueError):
    pass


def _clean(a) -> list:
    """ndarray -> nested lists with NaN -> None (JSON has no NaN)."""
    return np.where(np.isfinite(a), a, None).tolist() if a.dtype.kind == "f" else a.tolist()


def _num(s: str):
    try:
        return float(s)
    except ValueError:
        return s


class RunStore:
    def __init__(self, root) -> None:
        self.root = Path(root).resolve()

    # -- path safety: names are validated by regex AND must resolve to a direct child of the root
    def _run_dir(self, run_id: str) -> Path:
        if not isinstance(run_id, str) or not NAME.match(run_id):
            raise BadRequest("invalid run id")
        d = (self.root / run_id).resolve()
        if d.parent != self.root or not (d / "panel.npz").is_file():
            raise NotFound(f"no such run: {run_id}")
        return d

    def _panel(self, d: Path) -> dict:
        with np.load(d / "panel.npz") as z:
            return {k: z[k] for k in z.files}

    @staticmethod
    def _json(path: Path) -> dict:
        return json.loads(path.read_text()) if path.is_file() else {}

    def _label(self, d: Path) -> dict:
        meta = self._json(d / "meta.json")
        origin = meta.get("origin") if meta.get("origin") in ORIGINS else "unlabelled"
        return {"origin": origin, "description": str(meta.get("description", ""))}

    def list_runs(self) -> list:
        out = []
        for d in sorted(p for p in self.root.iterdir() if p.is_dir() and NAME.match(p.name)) if self.root.is_dir() else []:
            if (d / "panel.npz").is_file():
                out.append({"id": d.name, **self._label(d)})
        return out

    def run_info(self, run_id: str) -> dict:
        d = self._run_dir(run_id)
        panel = self._panel(d)
        manifest = self._json(d / "manifest.json")
        st = panel["status"]
        info = {
            "id": run_id, **self._label(d),
            "dates": [str(x) for x in panel["dates"].astype("datetime64[D]")],
            "tenor_days": panel["tenor_days"].tolist(), "k_nodes": panel["k_nodes"].tolist(),
            "status_labels": {str(k): v for k, v in STATUS_LABELS.items()},
            "status_counts": {v: int((st == k).sum()) for k, v in STATUS_LABELS.items()},
            "tables": sorted(p.stem for p in (d / "tables").glob("*.csv")) if (d / "tables").is_dir() else [],
            "forecasts": sorted(p.stem for p in (d / "forecasts").glob("*.npz")) if (d / "forecasts").is_dir() else [],
            "lineage": None,
        }
        if manifest:
            actual = panel_sha256(panel)
            info["lineage"] = {
                "version_id": manifest.get("version_id"), "config_sha256": manifest.get("config_sha256"),
                "inputs": manifest.get("inputs"), "ledgers_conserved": manifest.get("ledgers_conserved"),
                "ledger_prefilter": manifest.get("ledger_prefilter"), "ledger_quotes": manifest.get("ledger_quotes"),
                "panel_sha256_recorded": manifest.get("panel_sha256"), "panel_sha256_actual": actual,
                "panel_hash_matches": manifest.get("panel_sha256") == actual,
            }
        return info

    def surface(self, run_id: str, index: int) -> dict:
        d = self._run_dir(run_id)
        panel = self._panel(d)
        n = panel["iv"].shape[0]
        if not 0 <= index < n:
            raise NotFound(f"surface index out of range 0..{n - 1}")
        return {
            "run": run_id, **self._label(d), "index": index,
            "date": str(panel["dates"][index].astype("datetime64[D]")),
            "spot": float(panel["spot"][index]), "n_quotes": int(panel["n_quotes"][index]),
            "tenor_days": panel["tenor_days"].tolist(), "k_nodes": panel["k_nodes"].tolist(),
            "iv": _clean(panel["iv"][index]), "status": panel["status"][index].tolist(),
            "iv_units": "decimal (0.20 = 20%)",
        }

    def _forecast_file(self, d: Path, model: str) -> Path:
        if not isinstance(model, str) or not NAME.match(model):
            raise BadRequest("invalid forecast name")
        fp = d / "forecasts" / f"{model}.npz"
        if not fp.is_file():
            raise NotFound(f"no such forecast: {model}")
        return fp

    def forecast(self, run_id: str, model: str, index: int) -> dict:
        """Stored forecast of the surface at panel ``index`` next to the actual surface and their difference.

        ``forecasts/<model>.npz`` holds ``target_index`` (n,), ``origin_index`` (n,) and ``pred`` (n, T, K). The file is
        rejected if any origin is not strictly before its target (a forecast cannot use the future). The difference
        (forecast - actual) exists only where the actual cell is not missing and the forecast is finite; MAE is
        reported on cells whose actual status is OBSERVED, with the count.
        """
        d = self._run_dir(run_id)
        panel = self._panel(d)
        with np.load(self._forecast_file(d, model)) as z:
            tgt, org, pred = z["target_index"], z["origin_index"], z["pred"]
        n, T, K = panel["iv"].shape
        if pred.shape != (len(tgt), T, K) or org.shape != tgt.shape:
            raise BadRequest("forecast file shape does not match the run's panel")
        if (org >= tgt).any() or tgt.min(initial=0) < 0 or tgt.max(initial=0) >= n:
            raise BadRequest("forecast file has an origin at or after its target, or an index outside the panel")
        hit = np.flatnonzero(tgt == index)
        if hit.size != 1:
            raise NotFound(f"no forecast of this model for surface index {index}")
        f, o = pred[hit[0]], int(org[hit[0]])
        actual, st = panel["iv"][index], panel["status"][index]
        both = np.isfinite(f) & np.isfinite(actual) & (st != 0)
        diff = np.where(both, f - actual, np.nan)
        obs = both & (st == OBSERVED)
        return {
            "run": run_id, **self._label(d), "model": model, "index": index,
            "target_date": str(panel["dates"][index].astype("datetime64[D]")),
            "origin_date": str(panel["dates"][o].astype("datetime64[D]")), "horizon_panel_steps": index - o,
            "tenor_days": panel["tenor_days"].tolist(), "k_nodes": panel["k_nodes"].tolist(),
            "actual": _clean(np.where(st != 0, actual, np.nan)), "status": st.tolist(),
            "forecast": _clean(f), "difference": _clean(diff),
            "n_difference_cells": int(both.sum()), "n_observed_cells": int(obs.sum()),
            "mae_observed_vol_points": float(np.abs(diff[obs]).mean() * 100.0) if obs.any() else None,
            "iv_units": "decimal (0.20 = 20%); difference = forecast - actual",
        }

    def table(self, run_id: str, name: str) -> dict:
        d = self._run_dir(run_id)
        if not isinstance(name, str) or not NAME.match(name):
            raise BadRequest("invalid table name")
        fp = d / "tables" / f"{name}.csv"
        if not fp.is_file():
            raise NotFound(f"no such table: {name}")
        with open(fp, newline="") as fh:
            rows = list(csv.reader(fh))
        header, body = rows[0], rows[1:]
        return {"run": run_id, "table": name, "columns": header, "truncated": len(body) > MAX_TABLE_ROWS,
                "rows": [[_num(c) for c in r] for r in body[:MAX_TABLE_ROWS]]}


In [ ]:
%%writefile /content/nvs/src/volatility_lab/config/__init__.py
"""volatility_lab"""


In [ ]:
%%writefile /content/nvs/src/volatility_lab/config/settings.py
"""All tunable settings in one place (the notebook builds a ``Config`` and nothing else).

Every value is a *research setting for this experiment*, not an adopted product decision
(see decision.md DEC-012..DEC-015 and the experiment record in the run manifest).
"""
from __future__ import annotations

import dataclasses
import hashlib
import json
from dataclasses import dataclass, field
from typing import Tuple


@dataclass(frozen=True)
class DataConfig:
    kaggle_dataset: str = "dudesurfin/spy-options-eod-volatility-surface-2010-2023"
    # pre-filters applied while streaming batches (counted in the reject ledger)
    min_dte: int = 2
    max_dte: int = 400
    max_abs_log_strike: float = 0.45       # |ln(K/S)| pre-filter
    batch_rows: int = 500_000
    # per-quote filters (applied after the forward is known)
    min_mid: float = 0.05                  # USD; SPY tick is 0.01
    max_rel_spread: float = 0.35           # (ask-bid)/mid
    min_iv: float = 0.02
    max_iv: float = 3.0
    max_abs_k: float = 0.30                # |ln(K/F)| kept for the surface
    # forward / discount-factor estimation from put-call parity
    parity_window_k: float = 0.06          # near-ATM pairs used
    min_parity_pairs: int = 3
    regression_min_pairs: int = 5
    regression_min_T: float = 30 / 365
    df_bounds: Tuple[float, float] = (0.90, 1.01)
    forward_ratio_bounds: Tuple[float, float] = (0.85, 1.15)  # F/S plausibility


@dataclass(frozen=True)
class SurfaceConfig:
    tenor_days: Tuple[int, ...] = (7, 14, 30, 60, 91, 182)
    k_nodes: Tuple[float, ...] = (-0.20, -0.12, -0.08, -0.04, -0.02, 0.0, 0.02, 0.04, 0.08)
    min_quotes_per_slice: int = 5
    gap_tight: float = 0.02                # adjacent-quote gap in k treated as "observed"
    tol_obs_days: float = 1.0              # expiry within this many days of the tenor ...
    tol_obs_frac: float = 0.015            # ... or this fraction of the tenor (larger wins)


@dataclass(frozen=True)
class FeatureConfig:
    horizon: int = 1                       # forecast horizon in trading days (rows of the panel)
    lags: Tuple[int, ...] = (1, 5)
    ffill_days: int = 5
    seq_len: int = 20
    rv_windows: Tuple[int, ...] = (5, 20)
    rolling_mean_window: int = 20
    pca_components: int = 3


@dataclass(frozen=True)
class SplitConfig:
    first_test_year: int = 2016
    last_test_year: int = 2023
    min_train_years: int = 3               # years strictly before the validation year


@dataclass(frozen=True)
class TrainConfig:
    seeds: Tuple[int, ...] = (0, 1, 2)
    max_epochs: int = 150
    patience: int = 15
    batch_size: int = 128
    lr: float = 1e-3
    weight_decay: float = 1e-4
    hidden: int = 128
    gru_hidden: int = 64
    dropout: float = 0.1
    ridge_alphas: Tuple[float, ...] = (0.1, 1.0, 10.0, 100.0, 1000.0)
    models: Tuple[str, ...] = ("mlp", "gru")


@dataclass(frozen=True)
class Config:
    data: DataConfig = field(default_factory=DataConfig)
    surface: SurfaceConfig = field(default_factory=SurfaceConfig)
    features: FeatureConfig = field(default_factory=FeatureConfig)
    split: SplitConfig = field(default_factory=SplitConfig)
    train: TrainConfig = field(default_factory=TrainConfig)

    def to_dict(self) -> dict:
        return dataclasses.asdict(self)

    def sha256(self) -> str:
        blob = json.dumps(self.to_dict(), sort_keys=True, default=list).encode()
        return hashlib.sha256(blob).hexdigest()


In [ ]:
%%writefile /content/nvs/src/volatility_lab/datasets/__init__.py
"""volatility_lab"""


In [ ]:
%%writefile /content/nvs/src/volatility_lab/datasets/synthetic.py
"""SYNTHETIC option chains in the OptionsDX wide format, generated from a *known* volatility surface.

Used for CPU unit tests (known-answer IV / forward / surface recovery) and for a labelled plumbing run.
Never evidence about real markets. The latent surface is a smooth parametric function whose level, skew
and term slope follow independent AR(1) processes; the underlying is an independent GBM.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from volatility_lab.pricing.bs import black76_price

LABEL = "SYNTHETIC - not market data"


def true_sigma(k, T, level, skew, term=0.02, curv=1.5):
    return np.maximum(0.05, level + skew * k + curv * k * k + term * (T - 0.25))


def make_chain(n_days: int = 60, seed: int = 0, start: str = "2018-01-02", r: float = 0.02, q: float = 0.015,
               strike_step: float = 1.0, noise_free: bool = True, messy_format: bool = True,
               expiry_freq: str = "W-FRI", max_dte: int = 230):
    """Return ``(wide_frame, truth)``; ``truth`` holds per-date latent parameters, spot, F-factor and DF.

    ``expiry_freq`` is a pandas offset alias for the listed expiries ("W-FRI" weekly; "B" daily, as SPY now has).
    """
    rng = np.random.default_rng(seed)
    days = pd.bdate_range(start, periods=n_days)
    level, skew = 0.20, -0.8
    spot = 300.0
    fridays = pd.date_range(days[0], days[-1] + pd.Timedelta(days=max_dte), freq=expiry_freq)
    rows, truth = [], []
    for d in days:
        level = 0.20 + 0.97 * (level - 0.20) + 0.008 * rng.standard_normal()
        level = float(np.clip(level, 0.08, 0.6))
        skew = -0.8 + 0.98 * (skew + 0.8) + 0.03 * rng.standard_normal()
        spot *= float(np.exp(0.0003 + 0.01 * rng.standard_normal()))
        truth.append({"date": d, "level": level, "skew": skew, "spot": spot})
        for e in fridays:
            dte = (e - d).days
            if dte < 2 or dte > max_dte:
                continue
            T = dte / 365.0
            F = spot * np.exp((r - q) * T)
            df = np.exp(-r * T)
            K = np.arange(np.floor(0.72 * spot), np.ceil(1.28 * spot), strike_step * (1 if T < 0.3 else 5))
            sig = true_sigma(np.log(K / F), T, level, skew)
            call = df * black76_price(F, K, T, sig, True)
            put = df * black76_price(F, K, T, sig, False)
            half_c = np.maximum(0.005, 0.01 * call)
            half_p = np.maximum(0.005, 0.01 * put)
            if not noise_free:
                call, put = np.round(call, 2), np.round(put, 2)
            rows.append(pd.DataFrame({
                "[QUOTE_DATE]": d.strftime("%Y-%m-%d"), "[EXPIRE_DATE]": e.strftime("%Y-%m-%d"), "[STRIKE]": K,
                "[UNDERLYING_LAST]": spot, "[DTE]": float(dte),
                "[C_BID]": call - half_c, "[C_ASK]": call + half_c, "[P_BID]": put - half_p, "[P_ASK]": put + half_p,
                "[C_IV]": sig, "[P_IV]": sig}))
    wide = pd.concat(rows, ignore_index=True)
    # deep-OTM quotes with no bid, as in real chains
    wide.loc[wide["[C_BID]"] < 0.01, "[C_BID]"] = 0.0
    wide.loc[wide["[P_BID]"] < 0.01, "[P_BID]"] = 0.0
    if messy_format:  # padded strings like the raw vendor export
        for c in ("[STRIKE]", "[C_BID]", "[C_ASK]", "[P_BID]", "[P_ASK]", "[C_IV]", "[P_IV]", "[UNDERLYING_LAST]"):
            wide[c] = wide[c].map(lambda v: f" {v:.6f} ")
        wide["[QUOTE_DATE]"] = " " + wide["[QUOTE_DATE]"] + " "
        wide["[EXPIRE_DATE]"] = " " + wide["[EXPIRE_DATE]"] + " "
    return wide, pd.DataFrame(truth)


In [ ]:
%%writefile /content/nvs/src/volatility_lab/evaluation/__init__.py
"""volatility_lab"""


In [ ]:
%%writefile /content/nvs/src/volatility_lab/evaluation/compare.py
"""E-1 (nearest-expiry) versus E-2 (same-expiry) target comparison on identical (origin, cell) samples.

Both ``Prepared`` objects are built from the SAME surface panel, so they share origins, targets rows and the
walk-forward folds. Only the label of a cell differs:

* E-1: the value of whichever real expiry is nearest the tenor on the target date.
* E-2: the value of the expiry that anchored the cell on the ORIGIN date, tracked to the target date.

Everything here is descriptive: counts, errors, and shares. No test statistic is computed.
"""
from __future__ import annotations

from typing import Dict, List

import numpy as np
import pandas as pd

from volatility_lab.evaluation import metrics as M
from volatility_lab.evaluation.walkforward import Fold
from volatility_lab.features.panel import Prepared, target_arrays


def common_mask(p1: Prepared, p2: Prepared, positions) -> np.ndarray:
    """Cells that are evaluable under BOTH targets (primary = tight / observed on both)."""
    return M.eval_mask(p1, positions, "primary") & M.eval_mask(p2, positions, "primary")


def coverage_by_fold(p1: Prepared, p2: Prepared, folds: List[Fold]) -> pd.DataFrame:
    """Eligible-sample counts per test year under each target, their overlap, and E-2 censoring."""
    ty2, tst2 = target_arrays(p2)
    rows = []
    for f in folds:
        pos = f.test
        m1, m2 = M.eval_mask(p1, pos, "primary"), M.eval_mask(p2, pos, "primary")
        both = m1 & m2
        anchored = p2.status[p2.origins[pos]] >= 2          # origin cell anchored on one real expiry
        censored = anchored & (tst2[pos] == 0)               # ... but its expiry has no usable slice at the target
        sw = p1.anchor_changed[pos] if p1.anchor_changed is not None else np.zeros_like(m1)
        rows.append({"fold": f.test_year, "origin_cells": int(m1.size),
                     "e1_eligible": int(m1.sum()), "e2_eligible": int(m2.sum()), "common": int(both.sum()),
                     "e1_only": int((m1 & ~m2).sum()), "e2_only": int((m2 & ~m1).sum()),
                     "e2_anchored_at_origin": int(anchored.sum()), "e2_censored_no_target": int(censored.sum()),
                     "e1_eligible_anchor_switched": int((m1 & sw).sum()),
                     "e1_switch_share": float((m1 & sw).sum() / m1.sum()) if m1.sum() else np.nan})
    return pd.DataFrame(rows)


def _pool(g: pd.DataFrame) -> dict:
    n = g.n_cells.sum()
    return {"mae": float((g.mae * g.n_cells).sum() / n), "rmse": float(np.sqrt((g.rmse ** 2 * g.n_cells).sum() / n)),
            "n_cells": int(n), "n_days": int(g.n_days.sum())}


def paired_on_common(p1: Prepared, p2: Prepared, folds: List[Fold], preds1: Dict[int, Dict[str, np.ndarray]],
                     preds2: Dict[int, Dict[str, np.ndarray]]):
    """Per (fold, model, target) and pooled errors on the common sample. Models are those present in both."""
    per = []
    for f in folds:
        mask = common_mask(p1, p2, f.test)
        for name in preds1[f.test_year]:
            if name not in preds2[f.test_year]:
                continue
            for label, p, preds in (("nearest_expiry", p1, preds1), ("fixed_expiry", p2, preds2)):
                per.append({"fold": f.test_year, "model": name, "target": label,
                            **M.overall(p, f.test, preds[f.test_year][name], mask)})
    per = pd.DataFrame(per)
    pooled = pd.DataFrame([{"model": m, "target": t, **_pool(g)} for (m, t), g in per.groupby(["model", "target"])])
    ref = pooled[pooled.model == "persistence"].set_index("target").rmse
    pooled["mse_skill_vs_persistence"] = 1.0 - pooled.rmse ** 2 / pooled.target.map(ref) ** 2
    return per, pooled


def switch_breakdown(p1: Prepared, folds: List[Fold], preds1: Dict[int, Dict[str, np.ndarray]],
                     model: str = "persistence") -> pd.DataFrame:
    """E-1 error of ``model`` on its primary cells, split by whether the nearest-expiry anchor changed at the target."""
    if p1.anchor_changed is None:
        raise ValueError("panel was built without anchor expiries")
    ty, _ = target_arrays(p1)
    rows = []
    for f in folds:
        mask = M.eval_mask(p1, f.test, "primary")
        err = (preds1[f.test_year][model] - ty[f.test]) * M.PTS
        for name, sel in (("anchor_switched", p1.anchor_changed[f.test]), ("same_expiry", ~p1.anchor_changed[f.test])):
            s = M._stats(err[mask & sel])
            rows.append({"fold": f.test_year, "group": name, **s})
    df = pd.DataFrame(rows)
    out = []
    for name, g in df.groupby("group"):
        n = g.n_cells.sum()
        out.append({"group": name, "n_cells": int(n), "share": np.nan,
                    "mae": float((g.mae.fillna(0) * g.n_cells).sum() / n) if n else np.nan,
                    "rmse": float(np.sqrt((g.rmse.fillna(0) ** 2 * g.n_cells).sum() / n)) if n else np.nan})
    out = pd.DataFrame(out)
    out["share"] = out.n_cells / out.n_cells.sum()
    return out


In [ ]:
%%writefile /content/nvs/src/volatility_lab/evaluation/decision_rule.py
"""Pre-registered DESCRIPTIVE decision rule for "improves on persistence" wording (owner-approved 2026-10-16).

Source of truth: docs/research/PHASE4_NEURAL_EVIDENCE_REVIEW.md section 3. This is a checklist over numbers that
``runner.evaluate_all`` already produced; it is **not** a statistical test and produces no p-value or interval (L-22).

A model "improves on persistence under one label" only if ALL applicable criteria hold on the primary cells:
  a. pooled MSE skill vs persistence > 0 AND pooled MAE below persistence's;
  b. better than persistence by daily MAE in at least ``min_years_better`` test years;
  c. every seed's pooled MSE skill > 0 (``None`` = not applicable, e.g. deterministic baselines);
  d. skill > 0 in at least ``min_regimes`` realised-volatility regimes that each have n >= ``min_cells`` cells.
Criterion e is ``combine_labels``: it must hold under BOTH the nearest-expiry and the fixed-expiry label, otherwise
the result is reported as holding under one label only (never as a general improvement).
A criterion that cannot be evaluated is ``None`` (never silently passed).
"""
from __future__ import annotations

import numpy as np
import pandas as pd


def _regime_mse(brk: pd.DataFrame, model: str) -> pd.DataFrame:
    r = brk[(brk.group_type == "rv_regime") & (brk.model == model)]
    out = {}
    for g, d in r.groupby("group"):
        n = int(d.n_cells.sum())
        sse = float((d.rmse.fillna(0.0) ** 2 * d.n_cells).sum())
        out[str(g)] = (n, sse / n if n else np.nan)
    return pd.DataFrame(out, index=["n", "mse"]).T


def improvement_rule(pooled: pd.DataFrame, brk: pd.DataFrame, paired: pd.DataFrame, model: str, *,
                     min_years_better: int = 6, min_regimes: int = 2, min_cells: int = 100,
                     seed_skills=None) -> dict:
    """Evaluate criteria a-d for ``model`` under one label. Returns ``{"a": {...}, ..., "label_pass": bool | None}``."""
    pr = pooled[pooled.cells == "primary"].set_index("model")
    skill, mae_m, mae_p = float(pr.loc[model, "mse_skill_vs_persistence"]), float(pr.loc[model, "mae"]), float(pr.loc["persistence", "mae"])
    a = {"pass": bool(skill > 0 and mae_m < mae_p), "detail": f"MSE skill {skill:+.4f}, MAE {mae_m:.4f} vs persistence {mae_p:.4f}"}

    row = paired[paired.model == model]
    if row.empty:
        b = {"pass": None, "detail": "no paired daily comparison for this model"}
    else:
        better, total = (int(x) for x in str(row.iloc[0].folds_better_than_persistence).split("/"))
        b = ({"pass": None, "detail": f"{better}/{total} years better; fewer than {min_years_better} years evaluated"} if total < min_years_better
             else {"pass": better >= min_years_better, "detail": f"{better}/{total} test years better by daily MAE (need >= {min_years_better})"})

    if seed_skills is None:
        c = {"pass": None, "detail": "not applicable (deterministic model, no seeds)"}
    else:
        c = {"pass": bool(len(seed_skills) > 0 and all(s > 0 for s in seed_skills)),
             "detail": "per-seed pooled MSE skill: " + ", ".join(f"{s:+.4f}" for s in seed_skills)}

    rm, rp = _regime_mse(brk, model), _regime_mse(brk, "persistence")
    elig = [g for g in rm.index if g in rp.index and rm.loc[g, "n"] >= min_cells and rp.loc[g, "n"] >= min_cells
            and np.isfinite(rm.loc[g, "mse"]) and rp.loc[g, "mse"] > 0]
    sk = {g: 1.0 - rm.loc[g, "mse"] / rp.loc[g, "mse"] for g in elig}
    if len(elig) < min_regimes:
        d = {"pass": None, "detail": f"only {len(elig)} regime(s) with n >= {min_cells}; need {min_regimes}"}
    else:
        d = {"pass": sum(v > 0 for v in sk.values()) >= min_regimes,
             "detail": "regime MSE skill: " + ", ".join(f"{g} {v:+.4f}" for g, v in sorted(sk.items()))}

    crit = {"a": a, "b": b, "c": c, "d": d}
    if any(crit[k]["pass"] is False for k in "abcd"):
        label_pass = False
    elif any(crit[k]["pass"] is None for k in "abd"):
        label_pass = None
    else:
        label_pass = True
    return {**crit, "label_pass": label_pass}


def combine_labels(nearest: dict, fixed: dict) -> str:
    """Criterion e. ``nearest``/``fixed`` are ``improvement_rule`` results under the two target definitions."""
    n, f = nearest["label_pass"], fixed["label_pass"]
    if n is True and f is True:
        return "improves_on_persistence_under_both_labels"
    if n is True:
        return "holds_under_nearest_expiry_label_only"
    if f is True:
        return "holds_under_fixed_expiry_label_only"
    if n is None and f is None:
        return "not_evaluable"
    return "no_improvement_shown"


In [ ]:
%%writefile /content/nvs/src/volatility_lab/evaluation/leakage.py
"""Executable leakage audit: features at an origin must not change when anything AFTER it changes."""
from __future__ import annotations

import dataclasses

import numpy as np

from volatility_lab.config.settings import Config
from volatility_lab.features.panel import (Prepared, causal_fill, cell_means, realized_vol, sequence_features,
                                           tabular_features)


def perturb_after(p: Prepared, row: int, cfg: Config, seed: int = 0) -> Prepared:
    """Copy of ``p`` whose surfaces, status and spot AFTER ``row`` are replaced by random values."""
    rng = np.random.default_rng(seed)
    iv, spot, st = p.iv.copy(), p.spot.copy(), p.status.copy()
    iv[row + 1:] = rng.uniform(0.05, 0.9, iv[row + 1:].shape)
    spot[row + 1:] = rng.uniform(100, 900, spot[row + 1:].shape)
    st[row + 1:] = 3
    return dataclasses.replace(p, iv=iv, spot=spot, status=st, avail=st > 0,
                               rv=realized_vol(spot, cfg.features.rv_windows))


def feature_invariance_check(p: Prepared, cfg: Config, row: int | None = None) -> dict:
    """Return a report; ``passed`` is True iff every tabular and sequence feature at origins <= ``row`` is identical."""
    row = int(row if row is not None else p.origins[len(p.origins) // 2])
    q = perturb_after(p, row, cfg)
    rows = np.arange(0, row + 1)
    fv = cell_means(p, rows)
    same_fill = np.array_equal(fv, cell_means(q, rows))
    f1, f2 = causal_fill(p, cfg.features.ffill_days, fv), causal_fill(q, cfg.features.ffill_days, fv)
    sel = p.origins <= row
    tab = np.array_equal(tabular_features(p, f1, cfg.features)[sel], tabular_features(q, f2, cfg.features)[sel])
    seq = np.array_equal(sequence_features(p, f1, cfg.features)[sel], sequence_features(q, f2, cfg.features)[sel])
    return {"perturbed_after_row": row, "origins_checked": int(sel.sum()), "fill_values_identical": same_fill,
            "tabular_identical": tab, "sequence_identical": seq, "passed": bool(same_fill and tab and seq)}


In [ ]:
%%writefile /content/nvs/src/volatility_lab/evaluation/metrics.py
"""Masked forecast-error metrics in implied-vol *points* (1 point = 0.01 of IV).

Evaluation mask (identical for every model): the target cell holds a real value of the requested status
(``primary``: OBSERVED only; ``all``: observed or interpolated) AND the cell had a real value at the
forecast origin, so persistence is defined everywhere the mask is true. Missing is never imputed here.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from volatility_lab.features.panel import Prepared, target_arrays
from volatility_lab.surfaces.grid import OBSERVED

PTS = 100.0


def eval_mask(p: Prepared, positions: np.ndarray, kind: str = "primary") -> np.ndarray:
    ty, tst = target_arrays(p)
    y, st = ty[positions], tst[positions]
    avail_t = p.avail[p.origins[positions]]
    tgt = (st == OBSERVED) if kind == "primary" else (st > 0)
    return np.isfinite(y) & tgt & avail_t


def _stats(err: np.ndarray) -> dict:
    n = int(err.size)
    if n == 0:
        return {"mae": np.nan, "rmse": np.nan, "n_cells": 0}
    return {"mae": float(np.mean(np.abs(err))), "rmse": float(np.sqrt(np.mean(err ** 2))), "n_cells": n}


def overall(p: Prepared, positions, pred, mask) -> dict:
    y = target_arrays(p)[0][positions]
    err = (pred - y)[mask] * PTS
    out = _stats(err)
    out["n_days"] = int(mask.any(axis=1).sum())
    return out


def daily_mae(p: Prepared, positions, pred, mask) -> np.ndarray:
    """Per-origin-day mean absolute error (NaN where the day has no evaluable cell)."""
    y = target_arrays(p)[0][positions]
    e = np.where(mask, np.abs(pred - y) * PTS, np.nan)
    with np.errstate(invalid="ignore"):
        return np.nanmean(e, axis=1)


MIN_CELLS = 100  # research setting, not a derived threshold: groups with fewer evaluated cells are flagged


def breakdowns(p: Prepared, positions, pred, mask, rv_cuts, min_cells: int = MIN_CELLS) -> pd.DataFrame:
    """Rows: (group_type, group, mae, rmse, n_cells, insufficient_n). Groups: tenor, k, calendar year, regime.

    ``insufficient_n`` is True when ``n_cells < min_cells``; such rows are still returned, never dropped."""
    y = target_arrays(p)[0][positions]
    err = (pred - y) * PTS
    rows = []

    def add(gt, g, m):
        s = _stats(err[m])
        rows.append({"group_type": gt, "group": g, **s})

    for tau in np.unique(p.cell_tenor):
        add("tenor_days", int(tau), mask & (p.cell_tenor == tau)[None, :])
    for k in np.unique(p.cell_k):
        add("log_moneyness", float(k), mask & (p.cell_k == k)[None, :])
    yr = p.year[p.origins[positions]]
    for Y in np.unique(yr):
        add("origin_year", int(Y), mask & (yr == Y)[:, None])
    rv20 = p.rv[p.origins[positions], -1]
    lo, hi = rv_cuts
    for name, sel in (("low", rv20 <= lo), ("mid", (rv20 > lo) & (rv20 <= hi)), ("high", rv20 > hi)):
        add("rv_regime", name, mask & sel[:, None])
    out = pd.DataFrame(rows)
    out["insufficient_n"] = out["n_cells"] < min_cells
    return out


def rv_terciles(p: Prepared, train_positions) -> tuple:
    """Regime cut-offs from TRAINING origins only (longest realised-vol window)."""
    r = p.rv[p.origins[train_positions], -1]
    r = r[np.isfinite(r)]
    return float(np.quantile(r, 1 / 3)), float(np.quantile(r, 2 / 3))


In [ ]:
%%writefile /content/nvs/src/volatility_lab/evaluation/walkforward.py
"""Expanding-window walk-forward folds with purging.

For each test year ``Y``: validation = year ``Y-1`` (early stopping / ridge penalty only), training = all
years before the validation year. Training origins whose *target day* falls on or after the first
validation origin are purged (their label would otherwise be a validation-period value); validation
origins are purged the same way against the test period. The test year is never used for selection.
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import List

import numpy as np

from volatility_lab.config.settings import SplitConfig
from volatility_lab.features.panel import Prepared


@dataclass
class Fold:
    test_year: int
    val_year: int
    train: np.ndarray   # positions into p.origins
    val: np.ndarray
    test: np.ndarray
    train_rows_end: int  # last panel row any training label touches (for fitting fill values)


def make_folds(p: Prepared, cfg: SplitConfig, val_mode: str = "prev_year", val_frac: float = 0.2) -> List[Fold]:
    """``val_mode='prev_year'`` is E-1/E-2. ``'tail_frac'`` (experiment H2) validates on the chronologically last
    ``val_frac`` of the origins available before the test year (labels purged against the test period); training =
    the earlier origins whose labels end before the first validation origin. ``Fold.val_year`` is then only a label."""
    if val_mode not in ("prev_year", "tail_frac"):
        raise ValueError(f"unknown val_mode: {val_mode}")
    oyear = p.year[p.origins]
    odate = p.dates[p.origins]
    tdate = p.dates[p.targets]
    folds = []
    for Y in range(cfg.first_test_year, cfg.last_test_year + 1):
        test = np.flatnonzero(oyear == Y)
        if val_mode == "tail_frac":
            pool = np.flatnonzero((oyear < Y) & (tdate < odate[test].min())) if test.size else test
            if pool.size == 0:
                continue
            n_val = max(1, int(np.ceil(val_frac * pool.size)))
            val, train = pool[-n_val:], pool[:-n_val]
            train = train[tdate[train] < odate[val].min()]
            if len(np.unique(oyear[train])) < cfg.min_train_years or train.size == 0:
                continue
            folds.append(Fold(Y, Y - 1, train, val, test, int(p.targets[train].max())))
            continue
        val = np.flatnonzero(oyear == Y - 1)
        if test.size == 0 or val.size == 0:
            continue
        train = np.flatnonzero((oyear < Y - 1) & (tdate < odate[val].min()))
        val = val[tdate[val] < odate[test].min()]
        if len(np.unique(oyear[train])) < cfg.min_train_years or val.size == 0:
            continue
        folds.append(Fold(Y, Y - 1, train, val, test, int(p.targets[train].max())))
    if not folds:
        raise ValueError("no walk-forward fold satisfies the split configuration")
    return folds


def assert_no_overlap(p: Prepared, f: Fold) -> None:
    """Hard leakage check: label days and origins are chronologically separated across the three sets."""
    odate, tdate = p.dates[p.origins], p.dates[p.targets]
    assert tdate[f.train].max() < odate[f.val].min(), "train labels reach into validation"
    assert tdate[f.val].max() < odate[f.test].min(), "validation labels reach into test"
    assert odate[f.train].max() < odate[f.val].min() <= odate[f.val].max() < odate[f.test].min(), "origins not chronological"


In [ ]:
%%writefile /content/nvs/src/volatility_lab/experiments/__init__.py
"""volatility_lab"""


In [ ]:
%%writefile /content/nvs/src/volatility_lab/experiments/runner.py
"""Orchestration: hosted files -> surfaces -> baselines -> GPU neural models -> descriptive evaluation.

Everything here is deterministic given (data bytes, Config, seeds). Outputs go to a caller-chosen
``out_dir`` (a Colab runtime path, optionally mirrored to Drive by the notebook); nothing is written to the
repository. No inferential statistic is computed: comparisons are descriptive with explicit counts.
"""
from __future__ import annotations

import hashlib
import json
import platform
import subprocess
import sys
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd

from volatility_lab.config.settings import Config
from volatility_lab.evaluation import metrics as M
from volatility_lab.evaluation.walkforward import Fold, assert_no_overlap
from volatility_lab.features.panel import (Prepared, causal_fill, cell_means, sequence_features,
                                           tabular_features, target_arrays)
from volatility_lab.ingestion.optionsdx import discover_files, fingerprint_files, load_wide
from volatility_lab.models import baselines
from volatility_lab.surfaces.diagnostics import diagnostics_report
from volatility_lab.surfaces.grid import STATUS_LABELS, build_surface_panel, status_counts
from volatility_lab.validation.quotes import build_otm_quotes


# ----------------------------------------------------------------------------- data -> surfaces
def build_surfaces(root, cfg: Config, *, max_rows=None, fingerprint=True, log=print) -> dict:
    files = discover_files(root)
    log(f"files: {[f.name for f in files]}")
    fp = fingerprint_files(files) if fingerprint else None
    wide, ledger1 = load_wide(files, cfg.data, max_rows=max_rows)
    log(f"prefilter ledger: {ledger1.to_dict()}")
    quotes, ledger2, diag = build_otm_quotes(wide, cfg.data)
    log(f"quote ledger: {ledger2.to_dict()}")
    panel = build_surface_panel(quotes, cfg.surface)
    return {"panel": panel, "fingerprint": fp, "ledger_prefilter": ledger1.to_dict(), "ledger_quotes": ledger2.to_dict(),
            "quote_diagnostics": diag, "surface_status_counts": status_counts(panel),
            "arbitrage_diagnostics": diagnostics_report(panel), "n_wide_rows_after_prefilter": int(len(wide)),
            "quotes": quotes}


def save_panel(panel: dict, path) -> None:
    np.savez_compressed(path, **panel)


def load_panel(path) -> dict:
    with np.load(path) as z:
        return {k: z[k] for k in z.files}


def data_audit(built: dict) -> dict:
    q, panel = built["quotes"], built["panel"]
    d = panel["dates"].astype("datetime64[D]")
    years = d.astype("datetime64[Y]").astype(int) + 1970
    per_year = pd.Series(years).value_counts().sort_index().to_dict()
    st = panel["status"]
    return {
        "first_date": str(d[0]), "last_date": str(d[-1]), "n_surface_dates": int(len(d)),
        "surface_dates_per_year": {int(k): int(v) for k, v in per_year.items()},
        "max_calendar_gap_days": int(np.diff(panel["dates"]).max()),
        "n_quotes": int(len(q)), "quotes_per_date_median": float(np.median(panel["n_quotes"])),
        "cells_total": int(st.size),
        "cell_share": {STATUS_LABELS[c]: float((st == c).mean()) for c in (0, 1, 2, 3)},
        "cell_share_by_tenor_observed": {int(t): float((st[:, j, :] == 3).mean()) for j, t in enumerate(panel["tenor_days"])},
        "median_rel_spread": float(q.rel_spread.median()),
        "iv_percentiles_pts": {str(p): float(np.percentile(q.iv, p) * 100) for p in (1, 5, 50, 95, 99)},
    }


# ----------------------------------------------------------------------------- model runs
def _fold_inputs(p: Prepared, f: Fold, cfg: Config):
    fill = cell_means(p, np.arange(0, f.train_rows_end + 1))
    return causal_fill(p, cfg.features.ffill_days, fill)


def run_baselines(p: Prepared, folds: List[Fold], cfg: Config, log=print):
    preds, chosen = {}, {}
    for f in folds:
        assert_no_overlap(p, f)
        filled = _fold_inputs(p, f, cfg)
        preds[f.test_year], chosen[f.test_year] = baselines.predict_baselines(p, f, filled, cfg)
        log(f"baselines fold {f.test_year}: train={len(f.train)} val={len(f.val)} test={len(f.test)} { {k: v for k, v in chosen[f.test_year].items() if k.endswith('_alpha')} }")
    return preds, chosen


def pca_factor_targets(s_all, D, M_, train, k):
    """Factor-space targets for experiment H5: basis ``V`` (k, C) from the TRAINING surfaces only; ``D`` -> its projection
    (missing-label cells = no change), loss mask all True."""
    mean_s = s_all[train].mean(axis=0)
    V = np.linalg.svd(s_all[train] - mean_s, full_matrices=False)[2][:k]
    return np.where(M_, D, 0.0) @ V.T, np.ones((D.shape[0], k), dtype=bool), V


def run_neural(p: Prepared, folds: List[Fold], cfg: Config, out_dir, *, torch_module=None, log=print, pca_factors=None):
    """Train/predict every (fold, model, seed); resumes from files already present in ``out_dir``.

    ``pca_factors=K`` (experiment H5): the network predicts the K PCA-factor changes (basis fitted on the training
    surfaces of the fold only, as in the PCA-ridge baseline) instead of every cell; the cell forecast is
    ``s_t + delta_z @ V``. Cells without a label count as "no change". Loss weights are all ones."""
    from volatility_lab.models import neural  # lazy: needs PyTorch + CUDA
    device = neural.require_cuda(torch_module)  # raises before any work (or any file) when no GPU
    log(f"GPU: {device}")
    out_dir = Path(out_dir)
    for sub in ("checkpoints", "predictions", "histories"):
        (out_dir / sub).mkdir(parents=True, exist_ok=True)
    preds: Dict[int, Dict[str, np.ndarray]] = {}
    for f in folds:
        assert_no_overlap(p, f)
        filled = _fold_inputs(p, f, cfg)
        s_all = filled[p.origins]
        y_all, ystat = target_arrays(p)
        D = y_all - s_all
        M_ = (ystat > 0) & np.isfinite(D)
        if pca_factors:
            D, M_, V = pca_factor_targets(s_all, D, M_, f.train, pca_factors)
        feats = {}
        if "mlp" in cfg.train.models:
            feats["mlp"] = tabular_features(p, filled, cfg.features)
        if "gru" in cfg.train.models:
            feats["gru"] = sequence_features(p, filled, cfg.features)
        preds[f.test_year] = {}
        for kind in cfg.train.models:
            X = feats[kind]
            data = {"Xtr": X[f.train], "Dtr": D[f.train], "Mtr": M_[f.train], "Xva": X[f.val], "Dva": D[f.val],
                    "Mva": M_[f.val], "Xte": X[f.test]}
            for seed in cfg.train.seeds:
                tag = f"fold{f.test_year}_{kind}_seed{seed}"
                pf = out_dir / "predictions" / f"{tag}.npy"
                ck = out_dir / "checkpoints" / f"{tag}.pt"
                hf = out_dir / "histories" / f"{tag}.json"
                if pf.exists() and ck.exists() and hf.exists():
                    delta = np.load(pf)
                    log(f"resume: {tag}")
                else:
                    r = neural.train_and_predict(kind, data, cfg.train, seed, torch_module=torch_module,
                                                 ckpt_path=str(ck), log=log)
                    delta = r.pop("pred_delta")
                    np.save(pf, delta)
                    hf.write_text(json.dumps(r, indent=1))
                    log(f"{tag}: epochs={r['epochs_run']} best_epoch={r['best_epoch']} {r['seconds']}s peak_mem={r['peak_gpu_mem_mb']}MB")
                preds[f.test_year][f"{kind}_s{seed}"] = s_all[f.test] + (delta @ V if pca_factors else delta)
            stack = np.stack([preds[f.test_year][f"{kind}_s{s}"] for s in cfg.train.seeds])
            preds[f.test_year][f"{kind}_ens"] = stack.mean(axis=0)
    return preds, device


# ----------------------------------------------------------------------------- evaluation
def export_forecasts(p: Prepared, folds: List[Fold], preds: Dict[int, Dict[str, np.ndarray]], out_dir) -> List[str]:
    """Write ``out_dir/forecasts/<model>.npz`` (``target_index``, ``origin_index``, ``pred`` (n, T, K)) for the viewer API.

    Indices are panel rows. Forecasts are the out-of-sample test-fold predictions exactly as evaluated; nothing is
    re-fitted or re-computed here. ``origin_index < target_index`` always holds (horizon >= 1). Under the E-2 label the
    viewer's "actual" is still the surface at the target date, not the same-expiry label the model was scored on.
    """
    sub = Path(out_dir) / "forecasts"
    sub.mkdir(parents=True, exist_ok=True)
    names = sorted({n for f in folds for n in preds[f.test_year]})
    for name in names:
        fs = [f for f in folds if name in preds[f.test_year]]
        pos = np.concatenate([f.test for f in fs])
        pred = np.concatenate([preds[f.test_year][name] for f in fs]).reshape(len(pos), p.n_tenor, p.n_k)
        np.savez_compressed(sub / f"{name}.npz", target_index=p.targets[pos], origin_index=p.origins[pos], pred=pred)
    return names


def evaluate_all(p: Prepared, folds: List[Fold], preds: Dict[int, Dict[str, np.ndarray]]):
    """Return ``(per_fold, pooled, breakdown, paired)`` DataFrames. Descriptive only (no p-values/CIs)."""
    per_fold, brk, daily = [], [], {}
    for f in folds:
        cuts = M.rv_terciles(p, f.train)
        for kind in ("primary", "all"):
            mask = M.eval_mask(p, f.test, kind)
            for name, pred in preds[f.test_year].items():
                per_fold.append({"fold": f.test_year, "model": name, "cells": kind, **M.overall(p, f.test, pred, mask)})
                if kind == "primary":
                    b = M.breakdowns(p, f.test, pred, mask, cuts)
                    b["fold"], b["model"] = f.test_year, name
                    brk.append(b)
                    daily.setdefault(name, []).append((f.test_year, M.daily_mae(p, f.test, pred, mask)))
    per_fold = pd.DataFrame(per_fold)
    brk = pd.concat(brk, ignore_index=True)

    rows = []
    for (cells, model), g in per_fold.groupby(["cells", "model"]):
        n = g.n_cells.sum()
        rows.append({"cells": cells, "model": model, "mae": (g.mae * g.n_cells).sum() / n,
                     "rmse": float(np.sqrt((g.rmse ** 2 * g.n_cells).sum() / n)), "n_cells": int(n),
                     "n_days": int(g.n_days.sum()), "n_folds": int(g.fold.nunique())})
    pooled = pd.DataFrame(rows)
    ref = pooled[pooled.model == "persistence"].set_index("cells").rmse
    pooled["mse_skill_vs_persistence"] = 1.0 - (pooled.rmse ** 2) / (pooled.cells.map(ref) ** 2)

    base = dict(daily["persistence"])
    prow = []
    for name, lst in daily.items():
        if name == "persistence":
            continue
        diffs, better_folds, tot_folds = [], 0, 0
        for fy, d in lst:
            ok = np.isfinite(d) & np.isfinite(base[fy])
            if ok.any():
                diffs.append(d[ok] - base[fy][ok])
                tot_folds += 1
                better_folds += int(np.mean(d[ok]) < np.mean(base[fy][ok]))
        dd = np.concatenate(diffs) if diffs else np.array([])
        prow.append({"model": name, "n_days": int(dd.size),
                     "mean_daily_mae_diff_vs_persistence_pts": float(dd.mean()) if dd.size else np.nan,
                     "share_of_days_better": float((dd < 0).mean()) if dd.size else np.nan,
                     "folds_better_than_persistence": f"{better_folds}/{tot_folds}"})
    return per_fold, pooled, brk, pd.DataFrame(prow)


# ----------------------------------------------------------------------------- provenance
def environment_metadata() -> dict:
    meta = {"python": sys.version.split()[0], "platform": platform.platform(), "numpy": np.__version__,
            "pandas": pd.__version__}
    try:
        import torch  # noqa: PLC0415
        meta["torch"] = torch.__version__
        meta["cuda_available"] = bool(torch.cuda.is_available())
    except (ImportError, OSError):
        meta["torch"] = None
    try:
        meta["nvidia_smi"] = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version,utilization.gpu", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=20).stdout.strip()
    except (OSError, subprocess.SubprocessError):
        meta["nvidia_smi"] = None
    return meta


def write_manifest(out_dir, extra: dict) -> dict:
    """Hash every file under ``out_dir`` and write ``manifest.json`` (excluded from its own listing)."""
    out_dir = Path(out_dir)
    files = {}
    for fp in sorted(out_dir.rglob("*")):
        if fp.is_file() and fp.name != "manifest.json":
            files[str(fp.relative_to(out_dir))] = {"bytes": fp.stat().st_size,
                                                    "sha256": hashlib.sha256(fp.read_bytes()).hexdigest()}
    manifest = {**extra, "files": files}
    (out_dir / "manifest.json").write_text(json.dumps(manifest, indent=1, default=str))
    return manifest


def verify_manifest(out_dir) -> list:
    """Re-open and re-hash every file listed in the manifest; returns the list of problems (empty = OK)."""
    out_dir = Path(out_dir)
    m = json.loads((out_dir / "manifest.json").read_text())
    bad = []
    for rel, info in m["files"].items():
        fp = out_dir / rel
        if not fp.exists():
            bad.append(f"missing {rel}")
        elif hashlib.sha256(fp.read_bytes()).hexdigest() != info["sha256"]:
            bad.append(f"hash mismatch {rel}")
    return bad


In [ ]:
%%writefile /content/nvs/src/volatility_lab/features/__init__.py
"""volatility_lab"""


In [ ]:
%%writefile /content/nvs/src/volatility_lab/features/panel.py
"""Surface panel -> leakage-safe forecasting dataset.

Every feature at origin ``t`` uses only rows ``<= t``. Targets are the surface ``h`` panel rows ahead.
Anything *fitted* (cell means used to fill never-seen cells, scalers, PCA) is fitted by the caller on
training rows only and passed in.

Cell index ``c = tenor_index * n_k + k_index`` (tenor-major).
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional, Tuple

import numpy as np
import pandas as pd

from volatility_lab.config.settings import FeatureConfig


@dataclass
class Prepared:
    dates: np.ndarray        # (n,) epoch days
    year: np.ndarray         # (n,)
    spot: np.ndarray         # (n,)
    iv: np.ndarray           # (n, C) NaN where status == 0
    status: np.ndarray       # (n, C) int8
    avail: np.ndarray        # (n, C) bool: a real surface value exists on that day
    rv: np.ndarray           # (n, len(rv_windows)) annualised realised vol of the underlying, NaN early
    cell_tenor: np.ndarray   # (C,)
    cell_k: np.ndarray       # (C,)
    origins: np.ndarray      # (N,) row indices usable as forecast origins
    targets: np.ndarray      # (N,) row index of the target day
    horizon: int
    n_tenor: int
    n_k: int
    # Optional explicit target aligned with ``origins`` (N, C). ``None`` = the E-1 definition:
    # whatever the surface holds at ``targets``. Experiment E-2 fills them with the SAME-expiry value.
    target_iv: Optional[np.ndarray] = None
    target_status: Optional[np.ndarray] = None
    anchor_changed: Optional[np.ndarray] = None  # (N, C) bool: the nearest-expiry anchor differs at origin vs target
    target_name: str = "nearest_expiry"
    # Optional (n, n_tenor + 5) calendar block (experiment H3): days-to-expiry of the expiry anchoring each tenor on
    # that row (nominal tenor when none) and a Monday-Friday one-hot weekday. Row-local: uses only that row's date/quotes.
    calendar: Optional[np.ndarray] = None
    # Optional mean-reversion window (experiment H4): adds "surface minus its trailing ``mr_window``-row mean" to the
    # tabular / PCA features. Causal (rows <= t only). ``None`` = off.
    mr_window: Optional[int] = None

    @property
    def n_cells(self) -> int:
        return self.iv.shape[1]


def flatten_panel(panel: dict):
    n, nt, nk = panel["iv"].shape
    iv = panel["iv"].reshape(n, nt * nk)
    st = panel["status"].reshape(n, nt * nk)
    tenor = np.repeat(panel["tenor_days"], nk)
    k = np.tile(panel["k_nodes"], nt)
    return iv, st, tenor, k


def realized_vol(spot: np.ndarray, windows: Tuple[int, ...]) -> np.ndarray:
    """Annualised std of daily log returns over the last ``w`` rows (inclusive of today)."""
    r = pd.Series(np.log(spot)).diff()
    return np.column_stack([r.rolling(w).std().to_numpy() * np.sqrt(252.0) for w in windows])


def _flat_anchor(panel: dict) -> np.ndarray:
    """(n, C) expiry id anchoring each cell's tenor on that date (-1 = none); tenor-major like the cells."""
    return np.repeat(panel["anchor_expiry"], panel["k_nodes"].size, axis=1)


def anchor_switches(panel: dict, h: int) -> np.ndarray:
    """(n, C) bool, row t: the nearest-expiry anchor of the cell at t+h is a DIFFERENT expiry than at t."""
    a = _flat_anchor(panel)
    n = a.shape[0]
    sw = np.zeros(a.shape, dtype=bool)
    sw[: n - h] = (a[h:] != a[: n - h]) & (a[h:] >= 0) & (a[: n - h] >= 0)
    return sw


def fixed_expiry_targets(panel: dict, h: int):
    """Same-expiry target (experiment E-2). Returns ``(ty, tst)`` of shape (n, C), indexed by the ORIGIN row.

    For origin row ``t`` and a cell whose tenor is anchored on a real expiry ``E`` at ``t`` (status OBSERVED or
    INTERP_K), the target is the IV of THAT SAME expiry ``E`` at the same log-moneyness node on row ``t+h``.
    * The expiry is chosen from origin-date quotes only; the future is read only for the label itself.
    * If ``E`` has no usable slice on ``t+h`` (expired, too few quotes, node outside its quotes) the target is
      MISSING (status 0): censored, never imputed. Callers report the censoring counts.
    * status 3 = tight bracketing quotes on both dates, 2 = anchored but sparse on either date, 0 = missing.
    The cell's maturity shortens by ``h`` days between the two dates (a genuine roll-down effect).
    """
    n, nt, nk = panel["iv"].shape
    C = nt * nk
    anchor = _flat_anchor(panel)
    status = panel["status"].reshape(n, C)
    gap_tight = float(panel["gap_tight"]) if "gap_tight" in panel else 0.02
    lookup = {(int(r), int(e)): m for m, (r, e) in enumerate(zip(panel["slice_row"], panel["slice_expiry"]))}
    sv, sg = panel["slice_iv"], panel["slice_gap"]
    ty = np.full((n, C), np.nan)
    tst = np.zeros((n, C), dtype=np.int8)
    for t in range(n - h):
        for j in range(nt):
            e = anchor[t, j * nk]
            if e < 0:
                continue
            m = lookup.get((t + h, int(e)))
            if m is None:
                continue
            cols = slice(j * nk, (j + 1) * nk)
            v, g, o = sv[m], sg[m], status[t, cols]
            ok = np.isfinite(v) & (o >= 2)
            tight = ok & (o == 3) & (g <= gap_tight)
            ty[t, cols] = np.where(ok, v, np.nan)
            tst[t, cols] = np.where(tight, 3, np.where(ok, 2, 0))
    return ty, tst


def calendar_block(panel: dict) -> np.ndarray:
    """(n, n_tenor + 5): anchoring-expiry days-to-expiry per tenor, then weekday one-hot (Mon..Fri). Known at row t."""
    d = panel["dates"]
    dte = panel["anchor_expiry"].astype(float) - d[:, None]
    dte = np.where(panel["anchor_expiry"] >= 0, dte, panel["tenor_days"][None, :].astype(float))
    wd = (d + 3) % 7                       # epoch day 0 (1970-01-01) was a Thursday -> Monday = 0
    return np.hstack([dte, (wd[:, None] == np.arange(5)[None, :]).astype(float)])


def prepare(panel: dict, cfg: FeatureConfig, target: str = "nearest_expiry", calendar: bool = False,
            mr_window: Optional[int] = None) -> Prepared:
    if target not in ("nearest_expiry", "fixed_expiry"):
        raise ValueError(f"unknown target definition: {target}")
    iv, st, tenor, k = flatten_panel(panel)
    n = iv.shape[0]
    dates = panel["dates"]
    cal = dates.astype("datetime64[D]")
    year = cal.astype("datetime64[Y]").astype(int) + 1970
    rv = realized_vol(panel["spot"], cfg.rv_windows)
    min_hist = max(max(cfg.lags), cfg.seq_len - 1 + max(cfg.rv_windows))
    h = cfg.horizon
    t = np.arange(min_hist, n - h)
    gap_ok = (dates[t + h] - dates[t]) <= (h + 4)  # a missing stretch of days must not masquerade as "h days ahead"
    t = t[gap_ok]
    extra = {}
    if "anchor_expiry" in panel:
        extra["anchor_changed"] = anchor_switches(panel, h)[t]
    if target == "fixed_expiry":
        ty, tst = fixed_expiry_targets(panel, h)
        extra.update(target_iv=ty[t], target_status=tst[t])
    if calendar:
        extra["calendar"] = calendar_block(panel)
    if mr_window is not None:
        extra["mr_window"] = int(mr_window)
    return Prepared(dates=dates, year=year, spot=panel["spot"], iv=iv, status=st, avail=st > 0, rv=rv,
                    cell_tenor=tenor, cell_k=k, origins=t, targets=t + h, horizon=h,
                    n_tenor=len(panel["tenor_days"]), n_k=len(panel["k_nodes"]), target_name=target, **extra)


def cell_means(p: Prepared, rows: np.ndarray) -> np.ndarray:
    """Per-cell mean IV over ``rows`` (training rows only); cells never seen fall back to the global mean."""
    vals = np.where(p.avail[rows], p.iv[rows], np.nan)
    n = np.isfinite(vals).sum(axis=0)
    m = np.where(n > 0, np.nansum(vals, axis=0) / np.maximum(n, 1), np.nan)
    return np.where(np.isfinite(m), m, np.nanmean(m))


def causal_fill(p: Prepared, ffill_days: int, fill_values: np.ndarray) -> np.ndarray:
    """Forward-fill each cell for at most ``ffill_days`` rows (past only), then ``fill_values`` for the rest."""
    n, C = p.iv.shape
    cur = np.where(p.avail, p.iv, np.nan)
    out = np.empty((n, C))
    last = np.full(C, np.nan)
    age = np.full(C, np.inf)
    for t in range(n):
        has = np.isfinite(cur[t])
        last = np.where(has, cur[t], last)
        age = np.where(has, 0.0, age + 1.0)
        out[t] = np.where(age <= ffill_days, last, np.nan)
    return np.where(np.isfinite(out), out, fill_values[None, :])


def trailing_mean(a: np.ndarray, w: int) -> np.ndarray:
    """Mean of rows ``t-w+1..t`` (inclusive of t; shorter windows early on). Causal."""
    return pd.DataFrame(a).rolling(w, min_periods=1).mean().to_numpy()


def tabular_features(p: Prepared, filled: np.ndarray, cfg: FeatureConfig) -> np.ndarray:
    """(N, D): [s_t, s_t - s_{t-lag} for lag in lags, rv_t, availability fraction]. Rows = ``p.origins``."""
    t = p.origins
    parts = [filled[t]]
    for lag in cfg.lags:
        parts.append(filled[t] - filled[t - lag])
    parts.append(p.rv[t])
    parts.append(p.avail[t].mean(axis=1, keepdims=True))
    if p.calendar is not None:
        parts.append(p.calendar[t])
    if p.mr_window is not None:
        parts.append(filled[t] - trailing_mean(filled, p.mr_window)[t])
    return np.hstack(parts)


def sequence_features(p: Prepared, filled: np.ndarray, cfg: FeatureConfig) -> np.ndarray:
    """(N, L, 2C + n_rv): per day [filled surface, availability mask, realised vols] for rows t-L+1..t."""
    L = cfg.seq_len
    day = np.hstack([filled, p.avail.astype(float), np.nan_to_num(p.rv)] + ([p.calendar] if p.calendar is not None else []))
    idx = p.origins[:, None] + np.arange(-L + 1, 1)[None, :]
    return day[idx]


def target_arrays(p: Prepared):
    """``(y, y_status)`` aligned with ``p.origins``: target-day IV (NaN where missing) and its status."""
    if p.target_iv is not None:
        return p.target_iv, p.target_status
    return p.iv[p.targets], p.status[p.targets]


In [ ]:
%%writefile /content/nvs/src/volatility_lab/ingestion/__init__.py
"""volatility_lab"""


In [ ]:
%%writefile /content/nvs/src/volatility_lab/ingestion/optionsdx.py
"""Adapter for OptionsDX-style wide option-chain files (one row = one strike/expiry, call and put side by side).

Reads parquet or CSV in column-pruned batches (the full file is never held in memory) and returns a
compact numeric frame plus a *conserved* reject ledger: ``rows_in == rows_kept + sum(rejects)``.

Nothing here downloads data. ``discover_files`` only lists files that a hosted-data client (Kaggle in
the notebook) has already placed in runtime-local storage.
"""
from __future__ import annotations

import hashlib
from collections import Counter
from pathlib import Path
from typing import Iterable, List, Optional, Sequence

import numpy as np
import pandas as pd

from volatility_lab.config.settings import DataConfig

REQUIRED = ("quote_date", "expire_date", "strike", "underlying_last", "c_bid", "c_ask", "p_bid", "p_ask")
OPTIONAL = ("c_iv", "p_iv")
NUMERIC = ("strike", "underlying_last", "c_bid", "c_ask", "p_bid", "p_ask", "c_iv", "p_iv")

# Order matters: a row gets the *first* reason that applies (this is what makes the ledger conserve).
PREFILTER_REASONS = ("bad_quote_date", "bad_expiry", "bad_strike_or_spot", "dte_out_of_range",
                     "strike_far_from_spot", "no_usable_side")


class SchemaError(ValueError):
    """The file does not contain the columns the adapter needs."""


def normalize_name(name: str) -> str:
    return str(name).strip().strip("[]").strip().lower()


class RejectLedger:
    """Counts rows by terminal disposition; ``check()`` asserts conservation."""

    def __init__(self) -> None:
        self.rows_in = 0
        self.kept = 0
        self.rejects: Counter = Counter()

    def reject(self, reason: str, n: int) -> None:
        if n:
            self.rejects[reason] += int(n)

    def check(self) -> None:
        total = self.kept + sum(self.rejects.values())
        if total != self.rows_in:
            raise AssertionError(f"ledger not conserved: in={self.rows_in} kept+rejected={total}")

    def to_dict(self) -> dict:
        self.check()
        return {"rows_in": self.rows_in, "kept": self.kept, "rejects": dict(self.rejects)}


def discover_files(root) -> List[Path]:
    root = Path(root)
    files = sorted(root.rglob("*.parquet")) or sorted(root.rglob("*.csv"))
    if not files:
        raise FileNotFoundError(f"no .parquet or .csv files under {root}")
    return files


def fingerprint_files(paths: Sequence[Path]) -> dict:
    """Name, size and SHA-256 of every file (streamed). Identifies *which bytes* an experiment used."""
    out = []
    for p in paths:
        h = hashlib.sha256()
        with open(p, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 22), b""):
                h.update(chunk)
        out.append({"file": p.name, "bytes": p.stat().st_size, "sha256": h.hexdigest()})
    return {"files": out, "combined_sha256": hashlib.sha256("".join(f["sha256"] for f in out).encode()).hexdigest()}


def _to_date_days(col: pd.Series) -> np.ndarray:
    """Parse a date column (datetime or padded strings) to int days since epoch; NaT -> min int64."""
    if pd.api.types.is_datetime64_any_dtype(col):
        parsed = pd.to_datetime(col, errors="coerce")
    else:
        text = col.astype(str).str.strip()
        uniq = text.unique()
        mapping = dict(zip(uniq, pd.to_datetime(pd.Series(uniq), errors="coerce")))
        parsed = text.map(mapping)
    return parsed.values.astype("datetime64[D]").astype("int64")  # NaT -> iinfo.min


def prefilter_batch(df: pd.DataFrame, cfg: DataConfig, ledger: RejectLedger) -> pd.DataFrame:
    """Normalise one raw batch and drop rows that cannot be used, recording why."""
    df = df.rename(columns={c: normalize_name(c) for c in df.columns})
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise SchemaError(f"missing required columns: {missing}; have: {sorted(df.columns)}")
    n = len(df)
    ledger.rows_in += n
    num = {c: pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float) if c in df.columns
           else np.full(n, np.nan) for c in NUMERIC}
    d = _to_date_days(df["quote_date"])
    e = _to_date_days(df["expire_date"])
    nat = np.iinfo("int64").min
    dte = e - d
    strike, spot = num["strike"], num["underlying_last"]
    with np.errstate(invalid="ignore", divide="ignore"):
        pos = np.isfinite(strike) & np.isfinite(spot) & (strike > 0) & (spot > 0)
        far = np.abs(np.log(strike / spot)) > cfg.max_abs_log_strike
        c_ok = (num["c_bid"] > 0) & (num["c_ask"] > 0)
        p_ok = (num["p_bid"] > 0) & (num["p_ask"] > 0)
    conditions = [d == nat, e == nat, ~pos, (dte < cfg.min_dte) | (dte > cfg.max_dte), far, ~(c_ok | p_ok)]
    first = np.select(conditions, np.arange(1, len(conditions) + 1), default=0)
    for i, reason in enumerate(PREFILTER_REASONS, start=1):
        ledger.reject(reason, int((first == i).sum()))
    keep = first == 0
    ledger.kept += int(keep.sum())
    out = pd.DataFrame({"date": d[keep].astype(np.int32), "expiry": e[keep].astype(np.int32)})
    for c in ("strike", "underlying_last", "c_bid", "c_ask", "p_bid", "p_ask"):
        out[c] = num[c][keep]
    for c in OPTIONAL:
        out[c] = num[c][keep].astype(np.float32)
    return out.rename(columns={"underlying_last": "spot"})


def _iter_raw_batches(path: Path, batch_rows: int) -> Iterable[pd.DataFrame]:
    want = set(REQUIRED) | set(OPTIONAL)
    if path.suffix == ".parquet":
        import pyarrow.parquet as pq
        pf = pq.ParquetFile(path)
        cols = [n for n in pf.schema_arrow.names if normalize_name(n) in want]
        for batch in pf.iter_batches(batch_size=batch_rows, columns=cols):
            yield batch.to_pandas()
    else:
        for chunk in pd.read_csv(path, chunksize=batch_rows, usecols=lambda c: normalize_name(c) in want,
                                 dtype=str, skipinitialspace=True):
            yield chunk


def load_wide(paths: Sequence[Path], cfg: DataConfig, *, max_rows: Optional[int] = None):
    """Stream all files, returning ``(wide_frame, ledger)``. ``max_rows`` is for smoke tests only."""
    ledger = RejectLedger()
    parts = []
    seen = 0
    for p in paths:
        for raw in _iter_raw_batches(Path(p), cfg.batch_rows):
            if max_rows is not None and seen >= max_rows:
                break
            if max_rows is not None:
                raw = raw.iloc[: max_rows - seen]
            seen += len(raw)
            parts.append(prefilter_batch(raw, cfg, ledger))
    if not parts:
        raise SchemaError("no rows were read")
    wide = pd.concat(parts, ignore_index=True)
    ledger.check()
    return wide, ledger


In [ ]:
%%writefile /content/nvs/src/volatility_lab/models/__init__.py
"""volatility_lab"""


In [ ]:
%%writefile /content/nvs/src/volatility_lab/models/baselines.py
"""Non-neural forecasters on the surface panel. All fitting uses the fold's TRAINING origins only;
the ridge penalty is chosen on the validation year; the test year is only predicted.

* ``persistence``   : tomorrow's surface = today's (causally forward-filled) surface.
* ``rolling_mean``  : mean of the last ``rolling_mean_window`` filled surfaces (a mean-reversion anchor).
* ``ridge``         : per-cell ridge on [surface, lagged surface changes, realised vol, availability]
                      predicting the change ``y_{t+h} - s_t`` (only cells observed in the label contribute).
* ``pca_ridge``     : ridge on the same features projected on the first PCA factors of the training
                      surfaces, predicting factor changes; reconstruction ``s_t + dz @ V``.
"""
from __future__ import annotations

from typing import Dict, Tuple

import numpy as np
import pandas as pd

from volatility_lab.config.settings import Config
from volatility_lab.evaluation.metrics import eval_mask
from volatility_lab.evaluation.walkforward import Fold
from volatility_lab.features.panel import Prepared, tabular_features, target_arrays, trailing_mean


def standardize_fit(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    mu = X.mean(axis=0)
    sd = X.std(axis=0)
    return mu, np.where(sd < 1e-8, 1.0, sd)


def ridge_fit(X: np.ndarray, Y: np.ndarray, M: np.ndarray, alpha: float):
    """Independent ridge per output column using only rows where ``M[:, c]``. Returns ``(W, b)``."""
    D, C = X.shape[1], Y.shape[1]
    W = np.zeros((D, C))
    b = np.zeros(C)
    eye = np.eye(D)
    for c in range(C):
        m = M[:, c]
        if m.sum() < D + 1:
            b[c] = float(np.nanmean(Y[m, c])) if m.any() else 0.0
            continue
        Xm, ym = X[m], Y[m, c]
        xbar, ybar = Xm.mean(axis=0), ym.mean()
        Xc = Xm - xbar
        W[:, c] = np.linalg.solve(Xc.T @ Xc + alpha * eye, Xc.T @ (ym - ybar))
        b[c] = ybar - xbar @ W[:, c]
    return W, b


def _masked_mse(pred, p, positions):
    m = eval_mask(p, positions, "all")
    y = target_arrays(p)[0][positions]
    return float(np.mean((pred - y)[m] ** 2)) if m.any() else np.inf


def predict_baselines(p: Prepared, fold: Fold, filled: np.ndarray, cfg: Config) -> Tuple[Dict[str, np.ndarray], dict]:
    """Return ``({name: prediction[n_test, C]}, chosen_hyperparameters)``."""
    fc, tc = cfg.features, cfg.train
    org = p.origins
    s_all = filled[org]
    y_all, ystat = target_arrays(p)
    out: Dict[str, np.ndarray] = {}
    chosen: dict = {}

    out["persistence"] = s_all[fold.test].copy()
    rm = pd.DataFrame(filled).rolling(fc.rolling_mean_window, min_periods=fc.rolling_mean_window).mean().to_numpy()
    out["rolling_mean"] = rm[org][fold.test]

    X = tabular_features(p, filled, fc)
    mu, sd = standardize_fit(X[fold.train])
    Xs = (X - mu) / sd
    D_all = y_all - s_all                      # NaN where the target cell is missing
    M_train = (ystat[fold.train] > 0) & np.isfinite(D_all[fold.train])

    best = (np.inf, None, None)
    curve = {}
    for a in tc.ridge_alphas:
        W, b = ridge_fit(Xs[fold.train], D_all[fold.train], M_train, a)
        v = _masked_mse(s_all[fold.val] + Xs[fold.val] @ W + b, p, fold.val)
        curve[a] = v
        if v < best[0]:
            best = (v, a, (W, b))
    a, (W, b) = best[1], best[2]
    out["ridge"] = s_all[fold.test] + Xs[fold.test] @ W + b
    chosen["ridge_alpha"] = a
    chosen["ridge_val_mse"] = curve        # validation-year masked MSE per penalty (the selection curve)

    # PCA factor model on training surfaces
    S_tr = s_all[fold.train]
    mean_s = S_tr.mean(axis=0)
    _, _, vt = np.linalg.svd(S_tr - mean_s, full_matrices=False)
    V = vt[: fc.pca_components]                                   # (K, C)
    Z = (filled - mean_s) @ V.T                                  # (n, K) scores of every row
    z_t = Z[org]
    parts = [z_t] + [z_t - Z[org - lag] for lag in fc.lags] + [p.rv[org], p.avail[org].mean(axis=1, keepdims=True)]
    if p.calendar is not None:
        parts.append(p.calendar[org])
    if p.mr_window is not None:
        parts.append(z_t - trailing_mean(Z, p.mr_window)[org])
    Xz = np.hstack(parts)
    muz, sdz = standardize_fit(Xz[fold.train])
    Xzs = (Xz - muz) / sdz
    if p.target_iv is None:                                      # E-1: the filled surface at the target row
        dZ = Z[p.targets] - z_t
    else:                                                        # E-2: the tracked-expiry target; no-label cells = no change
        tgt = np.where(np.isfinite(D_all) & (ystat > 0), y_all, s_all)
        dZ = (tgt - mean_s) @ V.T - z_t
    ones = np.ones_like(dZ, dtype=bool)
    best = (np.inf, None, None)
    curve = {}
    for a in tc.ridge_alphas:
        Wz, bz = ridge_fit(Xzs[fold.train], dZ[fold.train], ones[fold.train], a)
        v = _masked_mse(s_all[fold.val] + (Xzs[fold.val] @ Wz + bz) @ V, p, fold.val)
        curve[a] = v
        if v < best[0]:
            best = (v, a, (Wz, bz))
    a, (Wz, bz) = best[1], best[2]
    out["pca_ridge"] = s_all[fold.test] + (Xzs[fold.test] @ Wz + bz) @ V
    chosen["pca_ridge_alpha"] = a
    chosen["pca_ridge_val_mse"] = curve
    return out, chosen


In [ ]:
%%writefile /content/nvs/src/volatility_lab/models/neural.py
"""GPU-only neural forecasters (compact MLP and GRU) for the surface-change target.

Compute policy (decision.md DEC-015): every tensor lives on a verified CUDA device and
``require_cuda`` raises -- it never falls back to CPU. ``torch`` is imported lazily so this module can be
imported (and the failure path unit-tested with a stand-in object) on machines without PyTorch.

Target: ``Delta = y_{t+h} - s_t`` divided by one global scalar (train std), so the masked MSE is
proportional to the vol-point MSE used in evaluation. Missing target cells carry zero loss weight.
"""
from __future__ import annotations

import copy
import time
from typing import Dict, Optional

import numpy as np

from volatility_lab.config.settings import TrainConfig

GPU_HELP = (
    "No CUDA GPU is available, so neural training/validation/inference is refused (no CPU fallback).\n"
    "In Colab: Runtime -> Change runtime type -> Hardware accelerator: GPU (T4 or better) -> Save, "
    "then Runtime -> Restart session and re-run from the top."
)


class GpuUnavailableError(RuntimeError):
    pass


def require_cuda(torch_module=None) -> dict:
    """Return device metadata or raise ``GpuUnavailableError``. ``torch_module`` lets tests inject a stub."""
    if torch_module is None:
        try:
            import torch as torch_module  # noqa: PLC0415
        except (ImportError, OSError) as exc:  # missing, or installed but unusable (e.g. broken DLLs)
            raise GpuUnavailableError(f"PyTorch could not be imported ({exc.__class__.__name__}).\n" + GPU_HELP) from exc
    if not torch_module.cuda.is_available():
        raise GpuUnavailableError(GPU_HELP)
    props = torch_module.cuda.get_device_properties(0)
    return {
        "device": "cuda:0", "name": props.name, "total_memory_gb": round(props.total_memory / 2 ** 30, 2),
        "capability": f"{props.major}.{props.minor}", "torch": torch_module.__version__,
        "cuda": torch_module.version.cuda, "cudnn": torch_module.backends.cudnn.version(),
    }


def _modules(torch):
    nn = torch.nn

    class MLP(nn.Module):
        def __init__(self, d_in, n_out, hidden, dropout):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(d_in, hidden), nn.GELU(), nn.Dropout(dropout),
                nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, n_out))

        def forward(self, x):
            return self.net(x)

    class GRUNet(nn.Module):
        def __init__(self, d_in, n_out, hidden, dropout):
            super().__init__()
            self.gru = nn.GRU(d_in, hidden, batch_first=True)
            self.drop = nn.Dropout(dropout)
            self.head = nn.Linear(hidden, n_out)

        def forward(self, x):
            out, _ = self.gru(x)
            return self.head(self.drop(out[:, -1]))

    return MLP, GRUNet


def _norm_fit(X: np.ndarray):
    axes = tuple(range(X.ndim - 1))
    mu = X.mean(axis=axes)
    sd = X.std(axis=axes)
    return mu, np.where(sd < 1e-8, 1.0, sd)


def train_and_predict(kind: str, data: Dict[str, np.ndarray], cfg: TrainConfig, seed: int, *,
                      torch_module=None, ckpt_path: Optional[str] = None, log=None) -> dict:
    """Train on ``data['Xtr']`` with early stopping on ``data['Xva']`` and predict ``data['Xte']``.

    ``data`` keys: Xtr, Dtr, Mtr, Xva, Dva, Mva, Xte (X: tabular (N,D) for ``mlp`` or sequences (N,L,D) for
    ``gru``; D*: change targets (NaN allowed); M*: boolean loss masks). Returns decimal-IV change
    predictions for the test rows plus history and metadata.
    """
    torch = torch_module
    if torch is None:
        import torch  # noqa: PLC0415
    info = require_cuda(torch)
    dev = torch.device("cuda:0")
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    MLP, GRUNet = _modules(torch)

    mu, sd = _norm_fit(data["Xtr"])
    ysd = float(np.sqrt(np.mean(data["Dtr"][data["Mtr"]] ** 2)))

    def X_(a):
        return torch.tensor(((a - mu) / sd), dtype=torch.float32, device=dev)

    def D_(a, m):
        return torch.tensor(np.where(m, a, 0.0) / ysd, dtype=torch.float32, device=dev), torch.tensor(m, dtype=torch.float32, device=dev)

    Xtr, Xva, Xte = X_(data["Xtr"]), X_(data["Xva"]), X_(data["Xte"])
    Dtr, Mtr = D_(data["Dtr"], data["Mtr"])
    Dva, Mva = D_(data["Dva"], data["Mva"])
    n_out = Dtr.shape[1]
    d_in = Xtr.shape[-1]
    net = (MLP if kind == "mlp" else GRUNet)(d_in, n_out, cfg.hidden if kind == "mlp" else cfg.gru_hidden, cfg.dropout).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    def masked_mse(pred, y, m):
        return ((pred - y) ** 2 * m).sum() / m.sum().clamp(min=1.0)

    hist = {"train_loss": [], "val_loss": []}
    best, best_state, best_epoch, bad = np.inf, None, -1, 0
    t0 = time.time()
    n = Xtr.shape[0]
    g = torch.Generator(device=dev)
    g.manual_seed(seed)
    for epoch in range(cfg.max_epochs):
        net.train()
        perm = torch.randperm(n, device=dev, generator=g)
        tot, cnt = 0.0, 0
        for i in range(0, n, cfg.batch_size):
            idx = perm[i:i + cfg.batch_size]
            loss = masked_mse(net(Xtr[idx]), Dtr[idx], Mtr[idx])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            tot += float(loss) * idx.numel()
            cnt += idx.numel()
        net.eval()
        with torch.no_grad():
            val = float(masked_mse(net(Xva), Dva, Mva))
        hist["train_loss"].append(tot / cnt)
        hist["val_loss"].append(val)
        if val < best - 1e-9:
            best, best_epoch, bad = val, epoch, 0
            best_state = copy.deepcopy({k: v.detach().clone() for k, v in net.state_dict().items()})
        else:
            bad += 1
            if bad >= cfg.patience:
                break
        if log and (epoch % 25 == 0):
            log(f"  [{kind} seed {seed}] epoch {epoch:3d} train {hist['train_loss'][-1]:.4f} val {val:.4f}")
    net.load_state_dict(best_state)
    net.eval()
    with torch.no_grad():
        pred = net(Xte).detach().cpu().numpy() * ysd
    out = {"pred_delta": pred, "history": hist, "best_epoch": int(best_epoch), "epochs_run": len(hist["val_loss"]),
           "best_val_loss_std_units": float(best), "target_scale": ysd, "seconds": round(time.time() - t0, 2),
           "peak_gpu_mem_mb": round(torch.cuda.max_memory_allocated() / 2 ** 20, 1), "device": info}
    if ckpt_path:
        torch.save({"state_dict": best_state, "kind": kind, "seed": seed, "d_in": d_in, "n_out": n_out,
                    "x_mu": mu, "x_sd": sd, "target_scale": ysd, "cfg": cfg.__dict__, "best_epoch": best_epoch},
                   ckpt_path)
    return out


In [ ]:
%%writefile /content/nvs/src/volatility_lab/pricing/__init__.py
"""volatility_lab"""


In [ ]:
%%writefile /content/nvs/src/volatility_lab/pricing/bs.py
"""Vectorised Black-76 pricing and implied-volatility inversion.

Conventions (mirror ``scripts/phase0_synthetic/phase0``, which is the scalar reference
implementation this module is cross-checked against in the tests):

* European option on a forward ``F``; ``T`` in years; prices are *undiscounted*
  (the caller divides market prices by the discount factor).
* A failed inversion is an explicit status, never a fabricated IV.
"""
from __future__ import annotations

import numpy as np
from scipy.special import ndtr

# int8 status codes (kept numeric so they live in arrays)
OK = 0
BELOW_LOWER_BOUND = 1  # price <= intrinsic: no positive IV
ABOVE_UPPER_BOUND = 2  # price >= call: F / put: K, no finite IV
INVALID_INPUT = 3
STATUS_NAMES = {OK: "OK", BELOW_LOWER_BOUND: "BELOW_LOWER_BOUND",
                ABOVE_UPPER_BOUND: "ABOVE_UPPER_BOUND", INVALID_INPUT: "INVALID_INPUT"}

SIGMA_LO = 1e-4
SIGMA_HI = 5.0


def _d1_d2(F, K, T, sigma):
    vol = sigma * np.sqrt(T)
    d1 = (np.log(F / K) + 0.5 * vol * vol) / vol
    return d1, d1 - vol


def black76_price(F, K, T, sigma, is_call):
    """Undiscounted Black-76 price; broadcasts over arrays. ``sigma`` must be > 0."""
    F, K, T, sigma = (np.asarray(x, dtype=float) for x in (F, K, T, sigma))
    is_call = np.asarray(is_call, dtype=bool)
    d1, d2 = _d1_d2(F, K, T, sigma)
    call = F * ndtr(d1) - K * ndtr(d2)
    put = K * ndtr(-d2) - F * ndtr(-d1)
    return np.where(is_call, call, put)


def black76_vega(F, K, T, sigma):
    """Undiscounted vega dP/dsigma (per 1.00 of vol, not per vol point)."""
    F, K, T, sigma = (np.asarray(x, dtype=float) for x in (F, K, T, sigma))
    d1, _ = _d1_d2(F, K, T, sigma)
    return F * np.exp(-0.5 * d1 * d1) / np.sqrt(2.0 * np.pi) * np.sqrt(T)


def black76_greeks(F, K, T, sigma, is_call):
    """Forward-measure sensitivities: ``(delta, gamma, vega)`` of the undiscounted price.

    ``delta`` is dP/dF, ``gamma`` d2P/dF2, ``vega`` dP/dsigma (per 1.00 of vol).
    Theta and rho are intentionally omitted: they need a rate/dividend model this
    pipeline does not claim to have.
    """
    F, K, T, sigma = (np.asarray(x, dtype=float) for x in (F, K, T, sigma))
    is_call = np.asarray(is_call, dtype=bool)
    d1, _ = _d1_d2(F, K, T, sigma)
    delta = np.where(is_call, ndtr(d1), ndtr(d1) - 1.0)
    gamma = np.exp(-0.5 * d1 * d1) / np.sqrt(2.0 * np.pi) / (F * sigma * np.sqrt(T))
    return delta, gamma, black76_vega(F, K, T, sigma)


def implied_vol(price, F, K, T, is_call, *, n_iter: int = 48, chunk: int = 1_000_000):
    """Vectorised IV by bisection on [SIGMA_LO, SIGMA_HI].

    Returns ``(iv, status)``; ``iv`` is NaN wherever ``status != OK``.
    48 halvings shrink the bracket to ~1e-13, below any market-data resolution.
    """
    price, F, K, T = (np.asarray(x, dtype=float) for x in (price, F, K, T))
    shape = np.broadcast_shapes(price.shape, F.shape, K.shape, T.shape, np.shape(is_call))
    p, f, k, t = (np.ravel(np.broadcast_to(x, shape)) for x in (price, F, K, T))
    c = np.ravel(np.broadcast_to(np.asarray(is_call, dtype=bool), shape))
    iv = np.full(p.size, np.nan)
    status = np.full(p.size, INVALID_INPUT, dtype=np.int8)

    with np.errstate(invalid="ignore"):
        ok_in = np.isfinite(p) & np.isfinite(f) & np.isfinite(k) & np.isfinite(t) & (f > 0) & (k > 0) & (t > 0) & (p >= 0)
        lower = np.where(c, np.maximum(f - k, 0.0), np.maximum(k - f, 0.0))
        upper = np.where(c, f, k)
        status[ok_in & (p <= lower)] = BELOW_LOWER_BOUND
        status[ok_in & (p >= upper)] = ABOVE_UPPER_BOUND
        solve = ok_in & (p > lower) & (p < upper)

    idx_all = np.flatnonzero(solve)
    for s in range(0, idx_all.size, chunk):
        idx = idx_all[s:s + chunk]
        ps, fs, ks, ts, cs = p[idx], f[idx], k[idx], t[idx], c[idx]
        lo = np.full(idx.size, SIGMA_LO)
        hi = np.full(idx.size, SIGMA_HI)
        for _ in range(n_iter):
            mid = 0.5 * (lo + hi)
            too_high = black76_price(fs, ks, ts, mid, cs) > ps
            hi = np.where(too_high, mid, hi)
            lo = np.where(too_high, lo, mid)
        sol = 0.5 * (lo + hi)
        # the root must lie inside the search bracket
        pl = black76_price(fs, ks, ts, np.full(idx.size, SIGMA_LO), cs)
        ph = black76_price(fs, ks, ts, np.full(idx.size, SIGMA_HI), cs)
        bracketed = (pl <= ps) & (ps <= ph)
        iv[idx] = np.where(bracketed, sol, np.nan)
        status[idx] = np.where(bracketed, OK, BELOW_LOWER_BOUND).astype(np.int8)
    return iv.reshape(shape), status.reshape(shape)


In [ ]:
%%writefile /content/nvs/src/volatility_lab/storage/__init__.py



In [ ]:
%%writefile /content/nvs/src/volatility_lab/storage/raw_store.py
"""Write-once raw store and content-versioned datasets (Phase 1; local filesystem only).

* Raw files are stored content-addressed (``raw/<sha[:2]>/<sha256><suffix>``): the same bytes always map to
  the same path, a path can never hold different bytes, and a corrected file is a *new* object.
* A dataset version is ``sha256(input hashes + config hash)``. Ingesting the same inputs with the same config
  yields the same version id and an identical manifest; any change yields a different id. Nothing is
  overwritten: an id that already exists must reproduce byte-identical manifest content or ingestion refuses.
* The manifest carries both reject ledgers (``rows_in == kept + rejects``, checked) and the panel hash.

Storage format is not a decision (DEC-009): this is a minimal local layout, not an adopted standard.
"""
from __future__ import annotations

import hashlib
import json
import shutil
from pathlib import Path
from typing import Sequence

import numpy as np

from volatility_lab.config.settings import Config
from volatility_lab.ingestion.optionsdx import load_wide
from volatility_lab.surfaces.grid import build_surface_panel, status_counts
from volatility_lab.validation.quotes import build_otm_quotes

SCHEMA = "dataset-1"


class OverwriteError(RuntimeError):
    """An existing object would be changed (raw bytes or a dataset manifest)."""


def sha256_file(path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 22), b""):
            h.update(chunk)
    return h.hexdigest()


def panel_sha256(panel: dict) -> str:
    """Hash of a surface panel's content (keys, dtypes, shapes, bytes); NaN-safe and file-format independent."""
    h = hashlib.sha256()
    for key in sorted(panel):
        a = np.ascontiguousarray(panel[key])
        h.update(f"{key}|{a.dtype.str}|{a.shape}|".encode())
        h.update(a.tobytes())
    return h.hexdigest()


class RawStore:
    def __init__(self, root) -> None:
        self.root = Path(root)

    def put(self, path) -> dict:
        """Store a file by content. Idempotent for identical bytes; refuses to alter an existing object."""
        path = Path(path)
        digest = sha256_file(path)
        dest = self.root / "raw" / digest[:2] / f"{digest}{path.suffix}"
        if dest.exists():
            if sha256_file(dest) != digest:
                raise OverwriteError(f"stored object {dest.name} no longer matches its hash")
        else:
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(path, dest)
        return {"name": path.name, "sha256": digest, "bytes": dest.stat().st_size, "stored": str(dest)}

    def verify(self) -> list:
        """Re-hash every stored object; returns problems (empty = intact)."""
        raw = self.root / "raw"
        bad = []
        for fp in sorted(raw.rglob("*")) if raw.exists() else []:
            if fp.is_file() and sha256_file(fp) != fp.stem:
                bad.append(f"hash mismatch {fp.name}")
        return bad


def dataset_version_id(input_hashes: Sequence[str], config_sha256: str) -> str:
    blob = json.dumps({"inputs": sorted(input_hashes), "config_sha256": config_sha256}, sort_keys=True)
    return hashlib.sha256(blob.encode()).hexdigest()


def ingest(root, paths: Sequence, cfg: Config, *, max_rows=None) -> dict:
    """Raw files -> stored raw objects -> quotes -> surface panel -> versioned dataset. Returns the manifest.

    ``manifest["reused"]`` is True when the identical dataset version already existed (nothing rewritten).
    """
    store = RawStore(root)
    inputs = [store.put(p) for p in sorted(map(Path, paths), key=lambda p: p.name)]
    version = dataset_version_id([i["sha256"] for i in inputs], cfg.sha256())
    wide, ledger1 = load_wide([Path(i["stored"]) for i in inputs], cfg.data, max_rows=max_rows)
    quotes, ledger2, diag = build_otm_quotes(wide, cfg.data)
    panel = build_surface_panel(quotes, cfg.surface)
    manifest = {
        "schema": SCHEMA, "version_id": version, "config_sha256": cfg.sha256(), "config": cfg.to_dict(),
        "max_rows": max_rows,
        "inputs": [{k: i[k] for k in ("name", "sha256", "bytes")} for i in inputs],
        "ledger_prefilter": ledger1.to_dict(), "ledger_quotes": ledger2.to_dict(), "ledgers_conserved": True,
        "quote_diagnostics": diag, "surface_status_counts": status_counts(panel), "panel_sha256": panel_sha256(panel),
    }
    text = json.dumps(manifest, indent=1, sort_keys=True, default=str)
    out = Path(root) / "datasets" / version
    mf = out / "manifest.json"
    if mf.exists():
        if mf.read_text() != text:
            raise OverwriteError(f"dataset {version[:12]} exists with different content; a correction needs a new version")
        return {**manifest, "reused": True}
    out.mkdir(parents=True)
    np.savez_compressed(out / "panel.npz", **panel)
    mf.write_text(text)
    return {**manifest, "reused": False}


In [ ]:
%%writefile /content/nvs/src/volatility_lab/surfaces/__init__.py
"""volatility_lab"""


In [ ]:
%%writefile /content/nvs/src/volatility_lab/surfaces/compare_methods.py
"""Phase 2 comparison of candidate interpolation methods and quote-filter policies (evidence for DEC-006).

This module is **research evidence code, not an adopted method**: ``surfaces.grid`` keeps its production
behaviour (linear in k, linear total variance in T). Nothing here decides DEC-006.

Criteria (fixed before looking at results; see docs/research/PHASE2_SURFACE_COMPARISON.md):

* **Leave-one-out in k**: for each interior quote of an expiry slice, hide it, predict its IV from the
  remaining quotes of the same slice, compare. Interior only, so no method ever extrapolates.
* **Leave-one-expiry-out in T**: hide a whole expiry, predict each of its quotes from the two neighbouring
  expiries (``thin`` steps either side), compare. Neighbour IV at the quote's k comes from the same linear-in-k
  rule for every T-method, so only the T-rule differs.
* Errors are reported in **vol points** (decimal IV x 100) by moneyness region with ``n``; a region below
  ``MIN_N`` is flagged ``insufficient_n`` instead of being ranked.
* Quote filters: kept-quote counts, surface cell-status shares, and reconstruction error of kept quotes and of
  non-missing cells against the *known* generating surface (synthetic data only).
"""
from __future__ import annotations

import dataclasses
from typing import Callable, Dict, List

import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline, PchipInterpolator

from volatility_lab.config.settings import Config, DataConfig
from volatility_lab.surfaces import grid

MIN_N = 100                      # per-region sample floor for ranking (research setting, not a Phase 0 threshold)
REGIONS = (("put_wing k<-0.10", -np.inf, -0.10), ("put_body -0.10<=k<-0.02", -0.10, -0.02),
           ("atm -0.02<=k<0.02", -0.02, 0.02), ("call_body 0.02<=k<0.10", 0.02, 0.10),
           ("call_wing k>=0.10", 0.10, np.inf))


# ---------------------------------------------------------------- candidate methods (k direction)
def k_linear_iv(k, iv, T, kq):
    return np.interp(kq, k, iv)


def k_linear_total_variance(k, iv, T, kq):
    return np.sqrt(np.interp(kq, k, iv * iv * T) / T)


def k_pchip_iv(k, iv, T, kq):
    return PchipInterpolator(k, iv, extrapolate=False)(kq)


def k_cubic_spline_iv(k, iv, T, kq):
    if k.size < 4:
        return np.full_like(np.asarray(kq, dtype=float), np.nan)
    return CubicSpline(k, iv, extrapolate=False)(kq)


K_METHODS: Dict[str, Callable] = {"linear_iv (production)": k_linear_iv, "linear_total_variance": k_linear_total_variance,
                                  "pchip_iv": k_pchip_iv, "cubic_spline_iv": k_cubic_spline_iv}


# ---------------------------------------------------------------- candidate methods (T direction)
def t_linear_total_variance(T, Tlo, Thi, ivlo, ivhi):
    w = (T - Tlo) / (Thi - Tlo)
    return np.sqrt(((1 - w) * ivlo ** 2 * Tlo + w * ivhi ** 2 * Thi) / T)


def t_linear_iv(T, Tlo, Thi, ivlo, ivhi):
    w = (T - Tlo) / (Thi - Tlo)
    return (1 - w) * ivlo + w * ivhi


def t_nearest_expiry(T, Tlo, Thi, ivlo, ivhi):
    return np.where(abs(T - Tlo) <= abs(Thi - T), ivlo, ivhi)


T_METHODS: Dict[str, Callable] = {"linear_total_variance (production)": t_linear_total_variance,
                                  "linear_iv": t_linear_iv, "nearest_expiry": t_nearest_expiry}


# ---------------------------------------------------------------- evaluation
def _slices(quotes: pd.DataFrame):
    for (d, e), s in quotes.groupby(["date", "expiry"], sort=True):
        s = s.sort_values("k")
        yield d, e, float(s["T"].iloc[0]), s.k.to_numpy(), s.iv.to_numpy()


def loo_k(quotes: pd.DataFrame, methods: Dict[str, Callable] = K_METHODS, *, min_pts: int = 8,
          stride: int = 3, date_stride: int = 1) -> pd.DataFrame:
    """One row per (method, held-out quote): ``k, gap, err`` (prediction - quote, decimal IV; NaN if undefined)."""
    dates = np.unique(quotes.date.to_numpy())[::date_stride]
    rows: List[tuple] = []
    for d, e, T, k, iv in _slices(quotes[quotes.date.isin(dates)]):
        if k.size < min_pts:
            continue
        for i in range(1, k.size - 1, stride):
            keep = np.arange(k.size) != i
            gap = k[i + 1] - k[i - 1]
            for name, f in methods.items():
                pred = float(f(k[keep], iv[keep], T, np.array([k[i]]))[0])
                rows.append((name, int(d), int(e), T, k[i], gap, pred - iv[i]))
    return pd.DataFrame(rows, columns=["method", "date", "expiry", "T", "k", "gap", "err"])


def loo_expiry_t(quotes: pd.DataFrame, methods: Dict[str, Callable] = T_METHODS, *, thin: int = 1,
                 min_pts: int = 8, stride: int = 4, date_stride: int = 1) -> pd.DataFrame:
    """Hide expiry j, predict its quotes from expiries j-thin and j+thin. ``thin`` widens the maturity gap."""
    dates = np.unique(quotes.date.to_numpy())[::date_stride]
    rows: List[tuple] = []
    for d, sub in quotes[quotes.date.isin(dates)].groupby("date", sort=True):
        sl = [(T, k, iv) for _, _, T, k, iv in _slices(sub) if k.size >= min_pts]
        for j in range(thin, len(sl) - thin):
            (Tl, kl, vl), (T, k, iv), (Th, kh, vh) = sl[j - thin], sl[j], sl[j + thin]
            inside = (k >= max(kl[0], kh[0])) & (k <= min(kl[-1], kh[-1]))
            idx = np.flatnonzero(inside)[::stride]
            if idx.size == 0:
                continue
            ivlo, ivhi = np.interp(k[idx], kl, vl), np.interp(k[idx], kh, vh)
            for name, f in methods.items():
                pred = np.asarray(f(T, Tl, Th, ivlo, ivhi), dtype=float)
                for q, p in zip(idx, pred):
                    rows.append((name, int(d), T, Th - Tl, k[q], float(p - iv[q])))
    return pd.DataFrame(rows, columns=["method", "date", "T", "t_gap", "k", "err"])


def summarize_by_region(df: pd.DataFrame, min_n: int = MIN_N) -> pd.DataFrame:
    """Per (method, region): n, n_missing, MAE/RMSE/p95 in vol points; ``insufficient_n`` below ``min_n``."""
    out = []
    for name, g in df.groupby("method", sort=False):
        for label, lo, hi in REGIONS + (("ALL", -np.inf, np.inf),):
            r = g[(g.k >= lo) & (g.k < hi)]
            e = r.err.to_numpy()
            ok = np.isfinite(e)
            a = np.abs(e[ok]) * 100.0
            out.append({"method": name, "region": label, "n": int(ok.sum()), "n_missing": int((~ok).sum()),
                        "mae_vp": float(a.mean()) if a.size else np.nan,
                        "rmse_vp": float(np.sqrt((a ** 2).mean())) if a.size else np.nan,
                        "p95_vp": float(np.quantile(a, 0.95)) if a.size else np.nan,
                        "insufficient_n": bool(ok.sum() < min_n)})
    return pd.DataFrame(out)


# ---------------------------------------------------------------- quote-filter policies (synthetic truth)
FILTER_POLICIES: Dict[str, dict] = {
    "default (min_mid 0.05, spread 0.35)": {},
    "strict (min_mid 0.25, spread 0.10)": {"min_mid": 0.25, "max_rel_spread": 0.10},
    "loose (min_mid 0.01, spread 0.80)": {"min_mid": 0.01, "max_rel_spread": 0.80},
}


def filter_policy_report(wide: pd.DataFrame, truth: pd.DataFrame, true_sigma: Callable, base: Config,
                         policies: Dict[str, dict] = FILTER_POLICIES) -> pd.DataFrame:
    """Run ``build_otm_quotes`` + the production surface builder under each policy and score against the truth.

    ``truth`` has per-date ``level``/``skew``; ``true_sigma(k, T, level, skew)`` is the generating surface.
    Errors are vs the *generating* surface, so this is only meaningful on synthetic data.
    """
    from volatility_lab.validation.quotes import build_otm_quotes
    lv = {int(np.datetime64(d, "D").astype("int64")): (L, S) for d, L, S in zip(truth["date"], truth["level"], truth["skew"])}
    rows = []
    for name, over in policies.items():
        dcfg: DataConfig = dataclasses.replace(base.data, **over)
        quotes, ledger, _ = build_otm_quotes(wide, dcfg)
        LS = np.array([lv[int(d)] for d in quotes.date.to_numpy()])
        qerr = np.abs(quotes.iv.to_numpy() - true_sigma(quotes.k.to_numpy(), quotes["T"].to_numpy(), LS[:, 0], LS[:, 1])) * 100
        panel = grid.build_surface_panel(quotes, base.surface)
        cell_err = []
        for i, d in enumerate(panel["dates"]):
            L, S = lv[int(d)]
            for j, tau in enumerate(panel["tenor_days"]):
                ok = panel["status"][i, j] > 0
                if ok.any():
                    t = true_sigma(panel["k_nodes"][ok], tau / 365.0, L, S)
                    cell_err.append(np.abs(panel["iv"][i, j][ok] - t) * 100)
        cell_err = np.concatenate(cell_err)
        st = grid.status_counts(panel)
        n_cells = int(panel["status"].size)
        rows.append({"policy": name, "rows_in": ledger.rows_in, "quotes_kept": ledger.kept,
                     "kept_share": ledger.kept / ledger.rows_in, "quote_mae_vp": float(qerr.mean()),
                     "quote_p95_vp": float(np.quantile(qerr, 0.95)), "quote_max_vp": float(qerr.max()),
                     **{f"cells_{k}": v / n_cells for k, v in st.items()},
                     "cell_mae_vp": float(cell_err.mean()), "cell_p95_vp": float(np.quantile(cell_err, 0.95)),
                     "cell_max_vp": float(cell_err.max()), "n_cells_scored": int(cell_err.size)})
    return pd.DataFrame(rows)


In [ ]:
%%writefile /content/nvs/src/volatility_lab/surfaces/diagnostics.py
"""Static-arbitrage *diagnostics* on the constant-maturity grid. These are necessary-condition flags only:
a surface passing them is not thereby arbitrage-free, and nothing is repaired or dropped.
"""
from __future__ import annotations

import numpy as np

from volatility_lab.pricing.bs import black76_price


def calendar_violations(iv: np.ndarray, tenor_days: np.ndarray, tol: float = 1e-4):
    """Total variance w = iv^2 * T must not decrease with maturity at fixed k. Returns (violations, checks)."""
    T = np.asarray(tenor_days, dtype=float) / 365.0
    w = iv ** 2 * T[None, :, None]
    dw = np.diff(w, axis=1)
    both = np.isfinite(dw)
    return int((dw[both] < -tol).sum()), int(both.sum())


def butterfly_violations(iv: np.ndarray, tenor_days: np.ndarray, k_nodes: np.ndarray, tol: float = 1e-6):
    """Undiscounted call price (F = 1) must be convex in strike. Returns (violations, checks)."""
    T = np.asarray(tenor_days, dtype=float) / 365.0
    K = np.exp(np.asarray(k_nodes, dtype=float))
    viol = checks = 0
    n, nt, nk = iv.shape
    for j in range(nt):
        sig = iv[:, j, :]
        with np.errstate(invalid="ignore"):
            price = black76_price(1.0, K[None, :], T[j], np.where(np.isfinite(sig), sig, 1.0), True)
        price = np.where(np.isfinite(sig), price, np.nan)
        slope = np.diff(price, axis=1) / np.diff(K)[None, :]
        curv = np.diff(slope, axis=1)
        ok = np.isfinite(curv)
        viol += int((curv[ok] < -tol).sum())
        checks += int(ok.sum())
    return viol, checks


def diagnostics_report(panel: dict) -> dict:
    cv, cc = calendar_violations(panel["iv"], panel["tenor_days"])
    bv, bc = butterfly_violations(panel["iv"], panel["tenor_days"], panel["k_nodes"])
    return {"calendar_violations": cv, "calendar_checks": cc, "calendar_rate": cv / cc if cc else None,
            "butterfly_violations": bv, "butterfly_checks": bc, "butterfly_rate": bv / bc if bc else None,
            "note": "necessary-condition flags on a coarse grid; not a proof of arbitrage-freedom"}


In [ ]:
%%writefile /content/nvs/src/volatility_lab/surfaces/grid.py
"""Constant-maturity implied-volatility surfaces on a fixed (tenor, log-moneyness) grid.

Cell status (the Q-25 fix: *observed* is defined by where real quotes are, not by grid coincidence):

* ``OBSERVED`` (3): an expiry lies within ``tol`` of the tenor AND the two real quotes bracketing the node
  in log-moneyness are closer than ``gap_tight``. The value is a linear interpolation *between those two
  adjacent real quotes of that one expiry* (second-order exact for strikes this dense).
* ``INTERP_K`` (2): same expiry anchor, but the bracketing quotes are far apart (sparse wing).
* ``INTERP_T`` (1): no expiry near the tenor; linear interpolation of total variance ``iv^2 * T`` between
  the two bracketing expiries.
* ``MISSING`` (0): outside the range of real quotes in k, or the tenor is not bracketed by expiries.
  **Nothing is ever extrapolated.**
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from volatility_lab.config.settings import SurfaceConfig

MISSING, INTERP_T, INTERP_K, OBSERVED = 0, 1, 2, 3
STATUS_LABELS = {MISSING: "missing", INTERP_T: "interpolated_T", INTERP_K: "interpolated_k", OBSERVED: "observed"}


def _slice_nodes(k: np.ndarray, iv: np.ndarray, nodes: np.ndarray):
    """Linear-in-k node values for one expiry slice; NaN outside [k.min, k.max]. Also the bracketing gap."""
    val = np.full(nodes.size, np.nan)
    gap = np.full(nodes.size, np.nan)
    inside = (nodes >= k[0]) & (nodes <= k[-1])
    if inside.any():
        val[inside] = np.interp(nodes[inside], k, iv)
        hi = np.clip(np.searchsorted(k, nodes[inside], side="left"), 1, k.size - 1)
        gap[inside] = k[hi] - k[hi - 1]
    return val, gap


def build_day_surface(day_q: pd.DataFrame, cfg: SurfaceConfig):
    """One date -> ``(iv[nt,nk], status[nt,nk])``."""
    iv, status, _ = build_day_surface_ex(day_q, cfg)
    return iv, status


def build_day_surface_ex(day_q: pd.DataFrame, cfg: SurfaceConfig):
    """One date -> ``(iv, status, extra)``.

    ``extra`` carries what a *tracked-expiry* target needs, all computed from this date's quotes only:
    ``anchor[nt]`` (expiry id of the real expiry anchoring each tenor, -1 if none) and the per-expiry slice
    arrays ``slice_expiry[M]``, ``slice_iv[M,nk]``, ``slice_gap[M,nk]``.
    """
    nodes = np.asarray(cfg.k_nodes, dtype=float)
    nt, nk = len(cfg.tenor_days), nodes.size
    exp_T, exp_id, vals, gaps = [], [], [], []
    for eid, s in day_q.groupby("expiry", sort=True):
        if len(s) < cfg.min_quotes_per_slice:
            continue
        k = s.k.to_numpy()
        order = np.argsort(k)
        v, g = _slice_nodes(k[order], s.iv.to_numpy()[order], nodes)
        exp_T.append(float(s["T"].iloc[0]))
        exp_id.append(int(eid))
        vals.append(v)
        gaps.append(g)
    iv = np.full((nt, nk), np.nan)
    status = np.zeros((nt, nk), dtype=np.int8)
    anchor = np.full(nt, -1, dtype=np.int64)
    if not exp_T:
        return iv, status, {"anchor": anchor, "slice_expiry": np.zeros(0, np.int64),
                            "slice_iv": np.zeros((0, nk)), "slice_gap": np.zeros((0, nk))}
    exp_T = np.asarray(exp_T)
    V = np.vstack(vals)
    G = np.vstack(gaps)
    valid = np.isfinite(V)
    for j, tau in enumerate(cfg.tenor_days):
        Tt = tau / 365.0
        tol = max(cfg.tol_obs_days / 365.0, cfg.tol_obs_frac * Tt)
        near = int(np.argmin(np.abs(exp_T - Tt)))
        anchored = abs(exp_T[near] - Tt) <= tol
        if anchored:
            anchor[j] = exp_id[near]
        for i in range(nk):
            if anchored and valid[near, i]:
                iv[j, i] = V[near, i]
                status[j, i] = OBSERVED if G[near, i] <= cfg.gap_tight else INTERP_K
                continue
            lo_idx = np.flatnonzero(valid[:, i] & (exp_T <= Tt))
            hi_idx = np.flatnonzero(valid[:, i] & (exp_T >= Tt))
            if lo_idx.size == 0 or hi_idx.size == 0:
                continue  # not bracketed: never extrapolate
            lo, hi = lo_idx[-1], hi_idx[0]
            if lo == hi:
                continue
            w = (Tt - exp_T[lo]) / (exp_T[hi] - exp_T[lo])
            var = (1 - w) * V[lo, i] ** 2 * exp_T[lo] + w * V[hi, i] ** 2 * exp_T[hi]
            iv[j, i] = np.sqrt(var / Tt)
            status[j, i] = INTERP_T
    return iv, status, {"anchor": anchor, "slice_expiry": np.asarray(exp_id, dtype=np.int64),
                        "slice_iv": V, "slice_gap": G}


def build_surface_panel(quotes: pd.DataFrame, cfg: SurfaceConfig) -> dict:
    """All dates -> arrays ``iv[n,nt,nk]`` (NaN where missing), ``status[n,nt,nk]``, ``dates`` (epoch days)."""
    dates, spots, ivs, sts, nq = [], [], [], [], []
    anchors, s_row, s_exp, s_iv, s_gap = [], [], [], [], []
    bounds = np.flatnonzero(np.diff(quotes.date.to_numpy()) != 0) + 1
    starts = np.r_[0, bounds]
    ends = np.r_[bounds, len(quotes)]
    for a, b in zip(starts, ends):
        day = quotes.iloc[a:b]
        iv, st, ex = build_day_surface_ex(day, cfg)
        if not (st > 0).any():
            continue
        row = len(dates)
        anchors.append(ex["anchor"])
        s_row.append(np.full(len(ex["slice_expiry"]), row, dtype=np.int64))
        s_exp.append(ex["slice_expiry"]); s_iv.append(ex["slice_iv"]); s_gap.append(ex["slice_gap"])
        dates.append(int(day.date.iloc[0]))
        spots.append(float(day.spot.iloc[0]))
        nq.append(len(day))
        ivs.append(iv)
        sts.append(st)
    if not dates:
        raise ValueError("no date produced a surface")
    return {"dates": np.asarray(dates, dtype=np.int64), "spot": np.asarray(spots), "n_quotes": np.asarray(nq),
            "iv": np.stack(ivs), "status": np.stack(sts),
            "gap_tight": np.asarray(cfg.gap_tight), "anchor_expiry": np.stack(anchors), "slice_row": np.concatenate(s_row), "slice_expiry": np.concatenate(s_exp),
            "slice_iv": np.vstack(s_iv), "slice_gap": np.vstack(s_gap),
            "tenor_days": np.asarray(cfg.tenor_days), "k_nodes": np.asarray(cfg.k_nodes)}


def status_counts(panel: dict) -> dict:
    st = panel["status"]
    return {STATUS_LABELS[c]: int((st == c).sum()) for c in (OBSERVED, INTERP_K, INTERP_T, MISSING)}


In [ ]:
%%writefile /content/nvs/src/volatility_lab/validation/__init__.py
"""volatility_lab"""


In [ ]:
%%writefile /content/nvs/src/volatility_lab/validation/quotes.py
"""Quote validation -> forward/discount factor -> OTM selection -> implied volatility.

Input is the compact wide frame from ``ingestion.optionsdx``. Output is one row per *kept* out-of-the-money
quote with its Black-76 implied volatility, plus a conserved ledger explaining every dropped row.

Method (assumptions are deliberate and listed in KNOWN_LIMITATIONS):

* The forward ``F`` and discount factor ``DF`` of each (date, expiry) slice come from put-call parity on
  near-ATM pairs: ``C - P = DF * (F - K)``. With >= ``regression_min_pairs`` pairs and a long enough
  maturity both are fitted by OLS; otherwise ``DF = 1`` and ``F`` is the median of ``C - P + K``.
  No external rate or dividend series is used.
* The underlying ETF options are American-style. Black-76 is a European model, so deep ITM quotes would be
  biased; only the OTM side of each strike is used, which keeps the early-exercise premium small.
* Quotes are mid prices of a closing snapshot, not executable prices.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from volatility_lab.config.settings import DataConfig
from volatility_lab.ingestion.optionsdx import RejectLedger
from volatility_lab.pricing import bs

QUOTE_REASONS = ("no_parity_forward", "implausible_forward", "k_out_of_range", "missing_or_no_bid",
                 "crossed_or_locked", "low_price", "wide_spread", "iv_below_lower_bound",
                 "iv_above_upper_bound", "iv_invalid_input", "iv_out_of_bounds")


def estimate_forwards(wide: pd.DataFrame, cfg: DataConfig) -> pd.DataFrame:
    """Per (date, expiry): ``F``, ``DF``, pair count, method and plausibility status.

    ``fstat``: 0 ok, 1 no usable parity pairs, 2 implausible forward vs spot.
    """
    c_ok = (wide.c_bid > 0) & (wide.c_ask > wide.c_bid)
    p_ok = (wide.p_bid > 0) & (wide.p_ask > wide.p_bid)
    with np.errstate(invalid="ignore", divide="ignore"):
        near = np.abs(np.log(wide.strike / wide.spot)) <= cfg.parity_window_k
    sel = c_ok & p_ok & near
    g = wide.loc[sel, ["date", "expiry", "strike", "spot"]].copy()
    g["D"] = 0.5 * (wide.c_bid[sel] + wide.c_ask[sel]) - 0.5 * (wide.p_bid[sel] + wide.p_ask[sel])
    g["KK"] = g.strike ** 2
    g["KD"] = g.strike * g.D
    g["F1"] = g.D + g.strike
    st = g.groupby(["date", "expiry"], sort=False).agg(
        n=("strike", "size"), sK=("strike", "sum"), sD=("D", "sum"), sKK=("KK", "sum"),
        sKD=("KD", "sum"), medF=("F1", "median"), spot=("spot", "mean"))
    idx = st.index.to_frame(index=False)
    T = (idx.expiry.to_numpy() - idx.date.to_numpy()) / 365.0
    n = st.n.to_numpy(dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        den = n * st.sKK.to_numpy() - st.sK.to_numpy() ** 2
        slope = (n * st.sKD.to_numpy() - st.sK.to_numpy() * st.sD.to_numpy()) / den
        icpt = (st.sD.to_numpy() - slope * st.sK.to_numpy()) / n
        df_reg = -slope
        f_reg = icpt / df_reg
    lo, hi = cfg.df_bounds
    use_reg = ((n >= cfg.regression_min_pairs) & (T >= cfg.regression_min_T) & (den > 0)
               & np.isfinite(df_reg) & np.isfinite(f_reg) & (df_reg >= lo) & (df_reg <= hi))
    DF = np.where(use_reg, df_reg, 1.0)
    F = np.where(use_reg, f_reg, st.medF.to_numpy())
    ratio = F / st.spot.to_numpy()
    rlo, rhi = cfg.forward_ratio_bounds
    fstat = np.where(n < cfg.min_parity_pairs, 1,
                     np.where((ratio < rlo) | (ratio > rhi) | ~np.isfinite(ratio), 2, 0))
    out = pd.DataFrame({"date": idx.date.to_numpy(), "expiry": idx.expiry.to_numpy(), "F": F, "DF": DF,
                        "n_pairs": n.astype(int), "regression": use_reg, "fstat": fstat.astype(np.int8)})
    return out.set_index(["date", "expiry"])


def build_otm_quotes(wide: pd.DataFrame, cfg: DataConfig):
    """Return ``(quotes, ledger, diagnostics)``; see module docstring."""
    ledger = RejectLedger()
    ledger.rows_in = len(wide)
    fw = estimate_forwards(wide, cfg)
    df = wide.join(fw[["F", "DF", "fstat"]], on=["date", "expiry"])
    F, K, S = df.F.to_numpy(), df.strike.to_numpy(), df.spot.to_numpy()
    T = (df.expiry.to_numpy() - df.date.to_numpy()) / 365.0
    fstat = df.fstat.to_numpy(dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        is_call = K >= F
        bid = np.where(is_call, df.c_bid, df.p_bid)
        ask = np.where(is_call, df.c_ask, df.p_ask)
        vend = np.where(is_call, df.c_iv, df.p_iv).astype(float)
        k = np.log(K / F)
        mid = 0.5 * (bid + ask)
        rel = (ask - bid) / mid
        conds = [~np.isfinite(fstat) | (fstat == 1),
                 fstat == 2,
                 ~(np.abs(k) <= cfg.max_abs_k),
                 ~((bid > 0) & (ask > 0)),
                 ask <= bid,
                 mid < cfg.min_mid,
                 rel > cfg.max_rel_spread]
    first = np.select(conds, np.arange(1, len(conds) + 1), default=0)
    for i, reason in enumerate(QUOTE_REASONS[:len(conds)], start=1):
        ledger.reject(reason, int((first == i).sum()))
    ok = first == 0

    DFv = df.DF.to_numpy()
    iv, status = bs.implied_vol(mid[ok] / DFv[ok], F[ok], K[ok], T[ok], is_call[ok])
    for code, reason in ((bs.BELOW_LOWER_BOUND, "iv_below_lower_bound"), (bs.ABOVE_UPPER_BOUND, "iv_above_upper_bound"),
                         (bs.INVALID_INPUT, "iv_invalid_input")):
        ledger.reject(reason, int((status == code).sum()))
    in_bounds = (status == bs.OK) & (iv >= cfg.min_iv) & (iv <= cfg.max_iv)
    ledger.reject("iv_out_of_bounds", int(((status == bs.OK) & ~in_bounds).sum()))
    ledger.kept = int(in_bounds.sum())

    sel = np.flatnonzero(ok)[in_bounds]
    quotes = pd.DataFrame({
        "date": df.date.to_numpy()[sel], "expiry": df.expiry.to_numpy()[sel], "T": T[sel],
        "strike": K[sel], "k": k[sel], "iv": iv[in_bounds], "is_call": is_call[sel], "mid": mid[sel],
        "rel_spread": rel[sel], "F": F[sel], "DF": DFv[sel], "spot": S[sel], "vendor_iv": vend[sel],
    }).sort_values(["date", "expiry", "strike"], kind="mergesort").reset_index(drop=True)
    ledger.check()

    vv = quotes.vendor_iv.to_numpy()
    good = np.isfinite(vv) & (vv > 1e-4)
    ours = quotes.iv.to_numpy()
    diag = {
        "forward_slices": int(len(fw)),
        "forward_slices_regression": int(fw.regression.sum()),
        "forward_slices_no_pairs": int((fw.fstat == 1).sum()),
        "forward_slices_implausible": int((fw.fstat == 2).sum()),
        "quotes_kept": int(len(quotes)),
        "vendor_iv_compared": int(good.sum()),
        # unit-agnostic: ~1.0 if the vendor IV is a decimal, ~100 if it is in percent
        "vendor_iv_median_ratio_vendor_over_ours": float(np.median(vv[good] / ours[good])) if good.any() else None,
        "vendor_iv_median_abs_diff_decimal": float(np.median(np.abs(vv[good] - ours[good]))) if good.any() else None,
    }
    return quotes, ledger, diag


In [ ]:
import sys, os, json, time, warnings, hashlib, dataclasses
sys.path.insert(0, "/content/nvs/src")
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200, "display.max_columns", 30)

from volatility_lab.config.settings import Config, DataConfig, SurfaceConfig, FeatureConfig, SplitConfig, TrainConfig
from volatility_lab.ingestion.optionsdx import discover_files, fingerprint_files
from volatility_lab.features.panel import prepare
from volatility_lab.evaluation.walkforward import make_folds, assert_no_overlap
from volatility_lab.evaluation.leakage import feature_invariance_check
from volatility_lab.experiments import runner
from volatility_lab.models import neural
from volatility_lab.surfaces import grid

cfg = Config(
    data=DataConfig(kaggle_dataset=KAGGLE_DATASET, min_dte=MIN_DTE, max_dte=MAX_DTE, min_mid=MIN_MID,
                    max_rel_spread=MAX_REL_SPREAD, max_abs_k=MAX_ABS_K),
    surface=SurfaceConfig(tenor_days=TENOR_DAYS, k_nodes=K_NODES),
    features=FeatureConfig(horizon=HORIZON, seq_len=SEQ_LEN),
    split=SplitConfig(first_test_year=FIRST_TEST_YEAR, last_test_year=LAST_TEST_YEAR, min_train_years=MIN_TRAIN_YEARS),
    train=TrainConfig(seeds=SEEDS, max_epochs=MAX_EPOCHS, patience=PATIENCE, batch_size=BATCH_SIZE, lr=LR,
                      weight_decay=WEIGHT_DECAY, models=MODELS, ridge_alphas=ALPHAS_H1))
print("package imported; config sha256:", cfg.sha256())


In [ ]:
# GPU smoke test on tiny RANDOM tensors: proves train / early-stop / predict / checkpoint on the GPU for both architectures.
# (Random data: this checks the code path only and says nothing about forecasting.)
if GPU_OK:
    import tempfile
    rng = np.random.default_rng(0)
    n, C, D, L = 300, 12, 40, 6
    smoke_cfg = TrainConfig(max_epochs=6, patience=3, batch_size=64)
    for kind, shape in (("mlp", (n, D)), ("gru", (n, L, D))):
        X = rng.normal(size=shape); Y = rng.normal(size=(n, C)) * 0.01; Mk = rng.random((n, C)) < 0.8
        data = {"Xtr": X[:200], "Dtr": Y[:200], "Mtr": Mk[:200], "Xva": X[200:250], "Dva": Y[200:250], "Mva": Mk[200:250], "Xte": X[250:]}
        with tempfile.TemporaryDirectory() as td:
            out = neural.train_and_predict(kind, data, smoke_cfg, seed=1, ckpt_path=str(Path(td) / "m.pt"))
            assert out["pred_delta"].shape == (50, C) and np.isfinite(out["pred_delta"]).all() and (Path(td) / "m.pt").exists()
        print(f"{kind}: OK on {out['device']['name']} | {out['epochs_run']} epochs | {out['seconds']}s | peak GPU {out['peak_gpu_mem_mb']} MB")
else:
    print("skipped: no CUDA GPU (see Section 3)")


## 5. Secure dataset connection
**Kaggle Notebook (also supported, no credentials needed):** run this same notebook on Kaggle with *+ Add Input* -> the dataset above, a GPU accelerator and `RUN_MODE = 'kaggle'`; the dataset is mounted read-only under `/kaggle/input` and outputs go to `/kaggle/working`.

**Colab + Kaggle API mode:** add your Kaggle credentials as *Colab Secrets* (left sidebar -> key icon): `KAGGLE_USERNAME` and `KAGGLE_KEY` (or a single `KAGGLE_API_TOKEN`), and switch *Notebook access* on for each. Credentials are read from Secrets only: never type them into a cell, and never commit or print them. The dataset is fetched by the official client into this runtime's **temporary cache** (not Drive, not your computer, not Git); the notebook then streams it in column-pruned batches.

**Data notes.** The uploader labels the dataset MIT, but the rows come from OptionsDX (a commercial vendor), whose own terms govern redistribution; they have not been verified. Use for private research only and do not redistribute the data or derived chain-level files.

In [ ]:
KAGGLE_HELP = ("Kaggle credentials not found in Colab Secrets.\n"
    "1) Create an API token at kaggle.com -> Settings -> API (yourself; do not paste it into a cell).\n"
    "2) In Colab: left sidebar -> key icon -> Add new secret: KAGGLE_USERNAME and KAGGLE_KEY (or KAGGLE_API_TOKEN); toggle 'Notebook access' on.\n"
    "3) Re-run this cell. (To test the pipeline without real data, set RUN_MODE = 'synthetic_plumbing'.)")

if RUN_MODE == "kaggle" and ON_KAGGLE:
    # Kaggle Notebook: attach the dataset with "+ Add Input" (Datasets -> search the id above). It is mounted read-only; nothing is downloaded or copied.
    root = Path("/kaggle/input"); slug = KAGGLE_DATASET.split("/")[-1]
    DATA_DIR = next((d for d in sorted(root.iterdir()) if d.is_dir() and slug in d.name), root)
    if not [*DATA_DIR.rglob("*.parquet"), *DATA_DIR.rglob("*.csv")]:
        raise RuntimeError(f"Attach the dataset to this Kaggle notebook (+ Add Input -> {KAGGLE_DATASET}); /kaggle/input holds: {[d.name for d in root.iterdir()]}")
    DATA_LABEL = f"REAL DATA: kaggle:{KAGGLE_DATASET} (attached input)"
    KAGGLE_VERSION = "attached input (version shown on the dataset page)"
elif RUN_MODE == "kaggle":
    try:
        from google.colab import userdata
        for name in ("KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"):
            try:
                os.environ[name] = userdata.get(name)
            except Exception:
                pass
    except ImportError:
        pass                                    # not in Colab: rely on environment variables
    if not (os.environ.get("KAGGLE_API_TOKEN") or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))):
        raise RuntimeError(KAGGLE_HELP)
    import kagglehub
    DATA_DIR = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    DATA_LABEL = f"REAL DATA: kaggle:{KAGGLE_DATASET}"
    KAGGLE_VERSION = DATA_DIR.name if DATA_DIR.parent.name == "versions" else "unknown"
elif RUN_MODE == "synthetic_plumbing":
    from volatility_lab.datasets.synthetic import make_chain, LABEL
    DATA_DIR = Path("/content/nvs_synth"); DATA_DIR.mkdir(exist_ok=True)
    raw, _ = make_chain(SYNTH_DAYS, seed=7, noise_free=False)
    raw.to_parquet(DATA_DIR / "synthetic_chain.parquet")
    DATA_LABEL, KAGGLE_VERSION = LABEL, "n/a"
else:
    raise ValueError("RUN_MODE must be 'kaggle' or 'synthetic_plumbing'")

files = discover_files(DATA_DIR)
FINGERPRINT = fingerprint_files(files)
print(DATA_LABEL, "| dataset version:", KAGGLE_VERSION)
for f in FINGERPRINT["files"]:
    print(f"  {f['file']:40s} {f['bytes']/2**20:9.1f} MiB  sha256 {f['sha256'][:16]}...")
print("files live in runtime-local storage only:", DATA_DIR)

RUN_ID = f"{RUN_MODE}_{cfg.sha256()[:8]}_{FINGERPRINT['combined_sha256'][:8]}"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
BASE_DIR = Path(DRIVE_DIR if USE_DRIVE else ("/kaggle/working/nvs_runs" if ON_KAGGLE else LOCAL_DIR))
OUT_DIR = BASE_DIR / RUN_ID
for sub in ("plots", "tables", "surfaces", "predictions", "checkpoints", "histories", "metadata"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)
print("run id:", RUN_ID, "\noutputs ->", OUT_DIR, "(resume: re-running with the same config and data reuses finished (fold, model, seed) files)")

def savefig(name):
    plt.savefig(OUT_DIR / "plots" / f"{name}.png", dpi=130, bbox_inches="tight")


## 6. Data audit
Streams the files in column-pruned batches, validates every row, and reports what was kept and why the rest was dropped. Every dropped row has exactly one reason (the ledger is conserved).

In [ ]:
import pyarrow.parquet as pq
t0 = time.time()
built = runner.build_surfaces(DATA_DIR, cfg, max_rows=MAX_ROWS, fingerprint=False)
panel, quotes = built["panel"], built["quotes"]
audit = runner.data_audit(built)
print(f"built in {time.time() - t0:.0f}s | {DATA_LABEL}")
if files[0].suffix == ".parquet":
    sch = pq.ParquetFile(files[0]).schema_arrow
    print("source schema:", [(n, str(t)) for n, t in zip(sch.names, sch.types)])
print(f"coverage: {audit['first_date']} -> {audit['last_date']} | surface dates {audit['n_surface_dates']} | max calendar gap {audit['max_calendar_gap_days']} days")
l1, l2 = built["ledger_prefilter"], built["ledger_quotes"]
print(f"\nstage 1 (streaming pre-filter): rows in {l1['rows_in']:,} -> kept {l1['kept']:,}")
display(pd.Series(l1["rejects"], name="rows").to_frame().assign(share=lambda d: d.rows / l1["rows_in"]))
print(f"stage 2 (one OTM quote per kept row): in {l2['rows_in']:,} -> usable IV quotes {l2['kept']:,}")
display(pd.Series(l2["rejects"], name="rows").to_frame().assign(share=lambda d: d.rows / l2["rows_in"]))
print("forward / vendor-IV diagnostics:"); display(pd.Series(built["quote_diagnostics"]).to_frame("value"))
print("quote mix:", {"call share": float(quotes.is_call.mean()), "duplicate (date,expiry,strike)": int(quotes.duplicated(["date", "expiry", "strike"]).sum()),
                     "median expiries per date": float(quotes.groupby("date").expiry.nunique().median()),
                     "DTE percentiles 5/50/95": [float(x) for x in np.percentile(quotes["T"] * 365, [5, 50, 95])]})
print("IV percentiles (vol pts):", audit["iv_percentiles_pts"], "| median relative spread:", round(audit["median_rel_spread"], 4))
display(pd.Series(audit["surface_dates_per_year"], name="surface dates").to_frame().T)
display(pd.Series(audit["cell_share"], name="share of grid cells").to_frame().T)
print("limitations visible here: closing-snapshot mids; SPY options are American-style (Black-76 is European: OTM quotes only); no external rate/dividend series (parity-implied).")


## 8. Targets, folds and leakage audit
Both `Prepared` datasets are built from the same panel, so origins, target rows and folds are identical; only the label differs. The expiry anchoring each cell is read from origin-date quotes; the future is read only to form the label.

In [ ]:
display = lambda x: print(x.to_string() if hasattr(x, "to_string") else x)   # plain-text tables: they land in the run log
from volatility_lab.evaluation import compare, metrics as Mx
from volatility_lab.features.panel import target_arrays
p1 = prepare(panel, cfg.features, "nearest_expiry", calendar=CALENDAR, mr_window=MR_WINDOW)      # E-1 label
p2 = prepare(panel, cfg.features, "fixed_expiry", calendar=CALENDAR, mr_window=MR_WINDOW)        # E-2 label
folds = make_folds(p1, cfg.split, VAL_MODE, VAL_FRAC)
assert np.array_equal(p1.origins, p2.origins) and np.array_equal(p1.targets, p2.targets)
rows = []
for f in folds:
    assert_no_overlap(p1, f)
    od = p1.dates[p1.origins].astype("datetime64[D]")
    rows.append({"test_year": f.test_year, "val_year": f.val_year, "train_from": str(od[f.train].min()), "train_to": str(od[f.train].max()),
                 "n_train": len(f.train), "n_val": len(f.val), "n_test": len(f.test)})
fold_table = pd.DataFrame(rows); display(fold_table)
LEAK = feature_invariance_check(p1, cfg)
print("leakage audit (features at origins <= t unchanged when every later row is scrambled):", LEAK); assert LEAK["passed"], "LEAKAGE DETECTED"
cov = compare.coverage_by_fold(p1, p2, folds)
print("eligible-sample coverage per test year (primary cells):"); display(cov)
tot = cov.sum(numeric_only=True)
print(f"TOTAL test cells: E-1 eligible {int(tot.e1_eligible):,} | E-2 eligible {int(tot.e2_eligible):,} | common {int(tot.common):,} | "
      f"E-2 anchored at origin {int(tot.e2_anchored_at_origin):,}, of which censored (expiry has no usable slice at the target) {int(tot.e2_censored_no_target):,} | "
      f"E-1 eligible cells whose nearest-expiry anchor SWITCHED: {int(tot.e1_eligible_anchor_switched):,} ({tot.e1_eligible_anchor_switched / tot.e1_eligible:.1%})")
FEATURE_DEFS = {"target_e1": "y(t+h,c) - s(t,c), y = value of the expiry nearest the tenor on t+h; eligible: observed on t+h and a real value on t",
                "target_e2": "y(t+h,c) - s(t,c), y = value on t+h of the SAME expiry that anchored c on t; eligible: tight on both dates; censored if that expiry has no usable slice on t+h",
                "features": "unchanged from E-1 (filled surface, lagged changes, realised vol, availability)",
                "fit_on_training_rows_only": ["per-cell fill means", "feature scalers", "PCA basis", "ridge penalty (validation year)", "early stopping (validation year)"]}
FEATURE_DEFS["variant"] = {"name": VARIANT, "val_mode": VAL_MODE, "val_frac": VAL_FRAC, "calendar_block": CALENDAR, "mr_window": MR_WINDOW, "pca_factors": PCA_FACTORS, "ridge_alphas": ALPHAS_H1}
print("variant:", FEATURE_DEFS["variant"], "| config sha256:", cfg.sha256())


## 9. Baseline models on both targets
Identical code, folds and settings; each model is trained and scored under its own target, then compared on the common sample.

In [ ]:
t0 = time.time()
preds1, ch1 = runner.run_baselines(p1, folds, cfg, log=lambda *_: None)
preds2, ch2 = runner.run_baselines(p2, folds, cfg, log=print)
print(f"baselines (both targets) done in {time.time() - t0:.0f}s; ridge penalties E-1: {[v['ridge_alpha'] for v in ch1.values()]} | E-2: {[v['ridge_alpha'] for v in ch2.values()]}")


## 10. GPU neural training on both targets
MLP and GRU, 3 seeds per (fold, model, target), early stopping on the validation year, the E-1 fixed hyper-parameters. Every tensor lives on the CUDA device; there is no CPU fallback.

In [ ]:
# H4: baselines only; the shared evaluation / plotting cells below expect these names
all1, all2 = preds1, preds2
DEVICE = {'name': 'none (baselines only, CPU)'}; NEURAL_SECONDS = 0.0
HIST = pd.DataFrame(columns=['target', 'run', 'epochs_run', 'best_epoch', 'seconds', 'peak_gpu_mb'])


## 11. Out-of-sample evaluation
Primary cells only (tight quotes). Errors in implied-vol points (1 pt = 0.01). Sample counts are printed with every table.

In [ ]:
per_fold1, pooled1, brk1, paired1 = runner.evaluate_all(p1, folds, all1)
per_fold2, pooled2, brk2, paired2 = runner.evaluate_all(p2, folds, all2)
def prim(pooled): return pooled[pooled.cells == "primary"].sort_values("rmse")
print("=== E-2 (fixed expiry) pooled out-of-sample, each model on its own eligible cells ==="); display(prim(pooled2).round(4))
print("=== E-1 (nearest expiry) re-run in this notebook, same models and settings ==="); display(prim(pooled1).round(4))

# reproduction check against the numbers published in docs/research/E1_RESULTS.md (Kaggle run kaggle_eaa46df0_76bf2f94)
E1_PUBLISHED_RMSE = {"pca_ridge": 2.5020, "persistence": 2.5686, "ridge": 2.5883, "gru_ens": 2.6659, "mlp_ens": 2.8219, "rolling_mean": 5.0162}
chk = prim(pooled1).set_index("model").rmse
repro = pd.DataFrame({"E1 published": pd.Series(E1_PUBLISHED_RMSE), "re-run here": chk.reindex(list(E1_PUBLISHED_RMSE)).round(4)}); repro["abs diff"] = (repro["re-run here"] - repro["E1 published"]).abs()
print("E-1 reproduction check (RMSE, vol pts):"); display(repro)
print("walk-forward RMSE by test year, E-2 fixed expiry (observed cells):"); display(per_fold2[per_fold2.cells == "primary"].pivot(index="fold", columns="model", values="rmse").round(3))
print("descriptive paired comparison vs persistence under E-2 (daily MAE; no p-values, no intervals):"); display(paired2.round(4))

# the head-to-head: identical (origin, cell) sample, each model under its own target
per_c, pooled_c = compare.paired_on_common(p1, p2, folds, all1, all2)
ens = [m for m in pooled_c.model.unique() if not m.endswith(("_s0", "_s1", "_s2"))]
tab = pooled_c[pooled_c.model.isin(ens)].pivot(index="model", columns="target", values=["rmse", "mse_skill_vs_persistence"]).round(4)
print(f"=== COMMON SAMPLE (cells eligible under both targets): n = {int(pooled_c.n_cells.iloc[0]):,} cells over {int(pooled_c.n_days.iloc[0]):,} origin days ==="); display(tab)
print("per-year RMSE on the common sample (persistence / pca_ridge / gru_ens / mlp_ens):")
display(per_c[per_c.model.isin(["persistence", "pca_ridge", "gru_ens", "mlp_ens"])].pivot_table(index="fold", columns=["model", "target"], values="rmse").round(3))
sb = compare.switch_breakdown(p1, folds, all1)
print("E-1 persistence error on its eligible cells, split by whether the nearest-expiry anchor switched between origin and target:"); display(sb.round(4))

rows = []
for f in folds:
    mask = compare.common_mask(p1, p2, f.test)
    for label, p, pr in (("nearest_expiry", p1, all1), ("fixed_expiry", p2, all2)):
        y = target_arrays(p)[0][f.test]
        for name in ("persistence", "pca_ridge"):
            err = (pr[f.test_year][name] - y) * 100
            for tau in TENOR_DAYS:
                m = mask & (p.cell_tenor == tau)[None, :]
                rows.append({"target": label, "model": name, "tenor_days": tau, "se": float((err[m] ** 2).sum()), "n": int(m.sum())})
tenor_c = pd.DataFrame(rows).groupby(["model", "target", "tenor_days"]).sum().reset_index(); tenor_c["rmse"] = np.sqrt(tenor_c.se / tenor_c.n)
print("RMSE by tenor on the common sample:"); display(tenor_c.pivot_table(index="tenor_days", columns=["model", "target"], values="rmse").round(3))
print("n cells by tenor on the common sample:", tenor_c[(tenor_c.model == "persistence") & (tenor_c.target == "fixed_expiry")].set_index("tenor_days").n.to_dict())

fig, axs = plt.subplots(1, 2, figsize=(14, 4))
for name, ls in (("persistence", "-"), ("pca_ridge", "--")):
    for label, col in (("nearest_expiry", "C0"), ("fixed_expiry", "C1")):
        d = tenor_c[(tenor_c.model == name) & (tenor_c.target == label)]; axs[0].plot(d.tenor_days, d.rmse, ls, color=col, marker="o", label=f"{name} | {label}")
axs[0].set_xlabel("tenor (days)"); axs[0].set_ylabel("RMSE (vol pts)"); axs[0].set_title("common sample: error by tenor"); axs[0].legend(fontsize=6)
yr = per_c[per_c.model.isin(["persistence", "pca_ridge"])].pivot_table(index="fold", columns=["model", "target"], values="rmse"); yr.plot(ax=axs[1], marker="o"); axs[1].set_title("common sample: RMSE by test year"); axs[1].legend(fontsize=6)
savefig("e2_tenor_and_year"); plt.show()


## 11a. Pre-registered decision rule
Applied to every non-persistence model under both labels. A criterion that cannot be evaluated is reported as such, never passed. Per-seed skills feed criterion (c).

In [ ]:
import re
from volatility_lab.evaluation.decision_rule import improvement_rule, combine_labels
def seed_skills(pooled, kind):
    pr = pooled[pooled.cells == "primary"].set_index("model")
    return [float(pr.loc[f"{kind}_s{s}", "mse_skill_vs_persistence"]) for s in cfg.train.seeds if f"{kind}_s{s}" in pr.index]
RULE = {}
for m in [x for x in prim(pooled2).model if x != "persistence" and not re.search(r"_s\d+$", x)]:
    kind = m[:-4] if m.endswith("_ens") else None
    r1 = improvement_rule(pooled1, brk1, paired1, m, min_years_better=MIN_YEARS_BETTER, min_regimes=MIN_REGIMES, min_cells=MIN_CELLS_REGIME, seed_skills=seed_skills(pooled1, kind) if kind else None)
    r2 = improvement_rule(pooled2, brk2, paired2, m, min_years_better=MIN_YEARS_BETTER, min_regimes=MIN_REGIMES, min_cells=MIN_CELLS_REGIME, seed_skills=seed_skills(pooled2, kind) if kind else None)
    RULE[m] = {"nearest_expiry": r1, "fixed_expiry": r2, "verdict": combine_labels(r1, r2)}
rule_rows = []
for m, v in RULE.items():
    for label in ("nearest_expiry", "fixed_expiry"):
        r = v[label]
        rule_rows.append({"model": m, "label": label, **{f"{k}_pass": r[k]["pass"] for k in "abcd"}, "label_pass": r["label_pass"], "verdict_both_labels": v["verdict"], **{f"{k}_detail": r[k]["detail"] for k in "abcd"}})
RULE_TABLE = pd.DataFrame(rule_rows)
print("=== decision rule (a pooled skill+MAE, b years better, c every seed, d regimes; None = not evaluable) ==="); display(RULE_TABLE[["model", "label", "a_pass", "b_pass", "c_pass", "d_pass", "label_pass", "verdict_both_labels"]])
for _, r in RULE_TABLE.iterrows():
    print(f"  {r.model:14s} {r.label:15s} a: {r.a_detail} | b: {r.b_detail} | d: {r.d_detail}")
print("VERDICTS:", {m: v["verdict"] for m, v in RULE.items()})
# the diagnostic H2 is about: the epoch early stopping selected (0-3 = effectively untrained)
BEST_EPOCH = HIST.assign(kind=HIST.run.str.extract(r"_(mlp|gru)_")[0], fold=HIST.run.str.extract(r"fold(\d+)_")[0].astype(int)).groupby(["target", "kind", "fold"]).best_epoch.agg(["min", "mean", "max"]).round(1)
print("best epoch per (target, model, fold):"); display(BEST_EPOCH)
NEAR_UNTRAINED = int((HIST.best_epoch <= 3).sum()); print(f"runs whose early stopping picked epoch <= 3 (effectively untrained): {NEAR_UNTRAINED} of {len(HIST)}")


## 11b. Extra curves and heatmaps for both targets
The E-1 extra-plot code (training curves for every fold and seed, per-cell error / bias / skill heatmaps, daily-error and cumulative curves, skill by year and seed spread, actual-vs-forecast surfaces) is run once per target; files are prefixed `e1_nearest_` and `e2_fixed_`. Under E-2 every model is scored, and every heatmap drawn, on its own same-expiry label. Descriptive only; a figure that fails is reported loudly and recorded in the run metadata, never skipped silently.

In [ ]:
# shared plotting code (identical to the E-1 notebook section 11b), stored as text and run once per target below
EXTRA_SRC = r'''import re as _re, warnings
from volatility_lab.evaluation import metrics as Mx
from volatility_lab.surfaces.grid import OBSERVED as _OBS
from volatility_lab.features.panel import target_arrays as _target_arrays
PLOT_TAG = globals().get('PLOT_TAG', ''); HIST_DIR = globals().get('HIST_DIR', OUT_DIR / 'histories')   # E-2 sets both per target
_sf = lambda n: savefig(PLOT_TAG + n)
_TY, _TST = _target_arrays(p)   # the label each model was scored on (E-1: surface at the target date; E-2: same-expiry value)
EXTRA_PLOT_ERRORS, PLOT_DATA, MIN_N = {}, {}, 30
NT, NK = p.n_tenor, p.n_k
tenors = p.cell_tenor.reshape(NT, NK)[:, 0]; ks = p.cell_k.reshape(NT, NK)[0]
ALL_MODELS = sorted({m for f in folds for m in preds_all[f.test_year]})
HM = [m for m in ALL_MODELS if not _re.search(r"_s\d+$", m)]          # baselines + ensembles
_dt = lambda x: pd.to_datetime(np.asarray(x).astype("datetime64[D]"))

def guard(name, fn):
    try:
        fn()
    except Exception as e:
        EXTRA_PLOT_ERRORS[name] = f"{type(e).__name__}: {e}"
        print(f"EXTRA PLOT FAILED: {name}: {EXTRA_PLOT_ERRORS[name]}")
    finally:
        plt.close("all")

def heat(ax, arr, title, cmap="viridis", vmin=None, vmax=None, fmt=None):
    im = ax.imshow(arr, aspect="auto", origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(NK)); ax.set_xticklabels([f"{k:.2f}" for k in ks], fontsize=6, rotation=90)
    ax.set_yticks(range(NT)); ax.set_yticklabels([str(int(t)) for t in tenors], fontsize=6)
    ax.set_xlabel("log-moneyness k"); ax.set_ylabel("tenor (days)"); ax.set_title(title, fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.03)

# ---- per-cell accumulators over all test folds (observed target cells with a real value at the origin)
acc = {m: {k: np.zeros(p.n_cells) for k in ("abs", "sq", "err", "n")} for m in ALL_MODELS}
for f in folds:
    mask = Mx.eval_mask(p, f.test, "primary"); y = _TY[f.test]
    for m in ALL_MODELS:
        raw = (preds_all[f.test_year][m] - y) * 100.0; ok = mask & np.isfinite(raw); e = np.where(ok, raw, 0.0)
        a = acc[m]; a["abs"] += np.abs(e).sum(0); a["sq"] += (e ** 2).sum(0); a["err"] += e.sum(0); a["n"] += ok.sum(0)
def cellmap(m, what):
    a = acc[m]; n = a["n"]
    with np.errstate(invalid="ignore", divide="ignore"):
        v = {"mae": a["abs"] / n, "bias": a["err"] / n, "rmse": np.sqrt(a["sq"] / n)}[what]
    return np.where(n >= MIN_N, v, np.nan).reshape(NT, NK)
for m in ALL_MODELS:
    for w in ("mae", "bias", "rmse"): PLOT_DATA[f"cell_{w}__{m}"] = cellmap(m, w)
    PLOT_DATA[f"cell_n__{m}"] = acc[m]["n"].reshape(NT, NK)

def fig_error_heatmaps():
    fig, axs = plt.subplots(3, len(HM), figsize=(3.2 * len(HM), 9), squeeze=False)
    vmax = np.nanpercentile(np.concatenate([cellmap(m, "mae").ravel() for m in HM]), 99)
    bmax = np.nanpercentile(np.abs(np.concatenate([cellmap(m, "bias").ravel() for m in HM])), 99)
    rmax = np.nanpercentile(np.concatenate([cellmap(x, "rmse").ravel() for x in HM]), 99)
    for j, m in enumerate(HM):
        heat(axs[0][j], cellmap(m, "mae"), f"{m}: MAE (vol pts)", "magma", 0, vmax)
        heat(axs[1][j], cellmap(m, "bias"), f"{m}: bias = pred - actual (vol pts)", "RdBu_r", -bmax, bmax)
        heat(axs[2][j], cellmap(m, "rmse"), f"{m}: RMSE (vol pts)", "magma", 0, rmax)
    fig.suptitle(f"Per-cell out-of-sample error, observed target cells, all test folds (cells with n < {MIN_N} blank)", fontsize=9)
    _sf("error_heatmaps"); plt.show()

def fig_skill_heatmaps():
    others = [m for m in HM if m != "persistence"]
    fig, axs = plt.subplots(1, len(others), figsize=(3.4 * len(others), 3.6), squeeze=False)
    for ax, m in zip(axs[0], others):
        with np.errstate(invalid="ignore", divide="ignore"):
            s = 1.0 - acc[m]["sq"] / acc["persistence"]["sq"]
        s = np.where((acc[m]["n"] >= MIN_N) & (acc["persistence"]["n"] >= MIN_N), s, np.nan).reshape(NT, NK); PLOT_DATA[f"cell_skill__{m}"] = s
        heat(ax, s, f"{m}: MSE skill vs persistence", "RdBu", -0.5, 0.5)
    fig.suptitle("Per-cell MSE skill vs persistence (blue = better than persistence, red = worse; descriptive, clipped at +/-0.5)", fontsize=9)
    _sf("skill_heatmaps"); plt.show()

def fig_level_heatmaps():
    obs = np.where(p.status == _OBS, p.iv, np.nan) * 100.0
    mean_, sd_ = pd.DataFrame(obs).mean().to_numpy().reshape(NT, NK), pd.DataFrame(obs).std().to_numpy().reshape(NT, NK)
    n_obs = (p.status == _OBS).sum(0).reshape(NT, NK)
    PLOT_DATA.update({"level_mean_iv_pts": mean_, "level_std_iv_pts": sd_, "level_observed_days": n_obs})
    fig, axs = plt.subplots(1, 4, figsize=(15, 3.6))
    heat(axs[0], mean_, "mean observed IV (vol pts)", "viridis"); heat(axs[1], sd_, "std of observed IV over time (vol pts)", "viridis")
    heat(axs[2], n_obs, "days with an observed value", "Blues")
    heat(axs[3], acc["persistence"]["n"].reshape(NT, NK), "evaluated test cells (persistence)", "Blues")
    _sf("level_and_count_heatmaps"); plt.show()

# ---- daily error series
DAILY = {}
def build_daily():
    dates, D = [], {m: [] for m in HM}
    for f in folds:
        mask = Mx.eval_mask(p, f.test, "primary"); dates.append(_dt(p.dates[p.targets[f.test]]))
        for m in HM: D[m].append(Mx.daily_mae(p, f.test, preds_all[f.test_year][m], mask))
    DAILY["dates"] = dates[0].append(dates[1:]) if len(dates) > 1 else dates[0]
    for m in HM: DAILY[m] = np.concatenate(D[m]); PLOT_DATA[f"daily_mae__{m}"] = DAILY[m]
    PLOT_DATA["daily_dates_epoch_days"] = DAILY["dates"].values.astype("datetime64[D]").astype(np.int64)
def fig_daily():
    fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
    for m in HM:
        both = np.isfinite(DAILY[m]) & np.isfinite(DAILY["persistence"]); diff = np.where(both, DAILY[m] - DAILY["persistence"], np.nan)
        axs[0].plot(DAILY["dates"], pd.Series(DAILY[m]).rolling(20, min_periods=5).mean().to_numpy(), lw=1, label=m)
        axs[1].plot(DAILY["dates"], pd.Series(diff).rolling(20, min_periods=5).mean().to_numpy(), lw=1, label=m)
        axs[2].plot(DAILY["dates"], np.cumsum(np.nan_to_num(diff)), lw=1, label=m)
    axs[0].set_ylabel("daily MAE (vol pts), 20-day mean"); axs[0].legend(fontsize=7, ncol=3); axs[0].set_title("daily out-of-sample MAE (observed cells)")
    axs[1].axhline(0, color="k", lw=0.6); axs[1].set_ylabel("MAE - persistence MAE (vol pts)"); axs[1].set_title("20-day mean daily MAE difference vs persistence (below 0 = better)")
    axs[2].axhline(0, color="k", lw=0.6); axs[2].set_ylabel("cumulative sum (vol pts)"); axs[2].set_title("cumulative daily MAE difference vs persistence (descriptive; gaps between test years are not bridged)")
    _sf("daily_error_curves"); plt.show()

# ---- breakdown curves
def fig_breakdowns():
    fig, axs = plt.subplots(1, 4, figsize=(19, 3.8))
    for ax, (gt, label) in zip(axs, (("tenor_days", "tenor (days)"), ("log_moneyness", "log-moneyness k"), ("origin_year", "origin year"), ("rv_regime", "realised-vol regime"))):
        t = pool_brk(brk[(brk.group_type == gt) & brk.model.isin(HM)].assign(group=lambda d: d.group.astype(str)))
        pv = t.pivot(index="group", columns="model", values="rmse")
        try: pv = pv.loc[sorted(pv.index, key=float)]
        except ValueError: pass
        pv.plot(ax=ax, marker="o", ms=3, lw=1); ax.set_xlabel(label); ax.set_ylabel("RMSE (vol pts)"); ax.legend(fontsize=6)
        PLOT_DATA[f"rmse_by_{gt}"] = pv.to_numpy(); PLOT_DATA[f"rmse_by_{gt}__index"] = np.array(pv.index.astype(str)); PLOT_DATA[f"rmse_by_{gt}__models"] = np.array(pv.columns.astype(str))
    _sf("rmse_breakdown_curves"); plt.show()

# ---- skill by year, seed spread
def fig_skill_and_seeds():
    pf = per_fold[per_fold.cells == "primary"].pivot(index="fold", columns="model", values="rmse")
    sk = 1.0 - (pf.div(pf["persistence"], axis=0)) ** 2
    prm = pooled[pooled.cells == "primary"].set_index("model").sort_values("mse_skill_vs_persistence")
    PLOT_DATA["skill_by_year"] = sk.to_numpy(); PLOT_DATA["skill_by_year__years"] = sk.index.to_numpy(); PLOT_DATA["skill_by_year__models"] = np.array(sk.columns.astype(str))
    fig, axs = plt.subplots(1, 3, figsize=(19, 4.2))
    sk[[m for m in sk.columns if m in HM and m != "persistence"]].plot(ax=axs[0], marker="o", lw=1); axs[0].axhline(0, color="k", lw=0.6)
    axs[0].set_ylabel("MSE skill vs persistence"); axs[0].set_title("walk-forward: skill by test year (>0 = better than persistence)")
    seed_cols = [m for m in sk.columns if m not in HM]
    if seed_cols:
        sk[seed_cols].plot(ax=axs[1], marker=".", lw=0.7); axs[1].axhline(0, color="k", lw=0.6)
    axs[1].set_ylabel("MSE skill vs persistence"); axs[1].set_title("per-seed skill by test year (seed-to-seed spread)")
    cols = ["tab:blue" if m in HM else "lightsteelblue" for m in prm.index]
    axs[2].barh(prm.index, prm.mse_skill_vs_persistence, color=cols); axs[2].axvline(0, color="k", lw=0.6)
    axs[2].set_xlabel("pooled MSE skill vs persistence"); axs[2].set_title("pooled skill, every model (dark = baseline / ensemble)")
    _sf("skill_by_year_and_seed_spread"); plt.show()

# ---- training curves, every fold x seed (the history JSON holds only train / validation loss)
def fig_training():
    rows = []
    for kind in MODELS:
        files = sorted(HIST_DIR.glob(f"fold*_{kind}_seed*.json"))
        if not files: continue
        years = sorted({int(_re.match(r"fold(\d+)_", fp.stem).group(1)) for fp in files}); ncol = 4; nrow = int(np.ceil(len(years) / ncol))
        fig, axs = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.2 * nrow), squeeze=False)
        for ax in axs.ravel()[len(years):]: ax.axis("off")
        for ax, yr in zip(axs.ravel(), years):
            for i, fp in enumerate(sorted(x for x in files if x.stem.startswith(f"fold{yr}_"))):
                h = json.loads(fp.read_text()); c = f"C{i}"; sd = int(_re.search(r"seed(\d+)", fp.stem).group(1))
                ax.plot(h["history"]["val_loss"], color=c, lw=1.1, label=f"val s{sd}"); ax.plot(h["history"]["train_loss"], color=c, lw=0.7, ls=":", label=f"train s{sd}")
                ax.axvline(h["best_epoch"], color=c, lw=0.6, ls="--"); ax.plot(h["best_epoch"], h["best_val_loss_std_units"], "o", color=c, ms=4)
                PLOT_DATA[f"history__{fp.stem}__val"] = np.array(h["history"]["val_loss"]); PLOT_DATA[f"history__{fp.stem}__train"] = np.array(h["history"]["train_loss"])
                rows.append({"run": fp.stem, "kind": kind, "fold": yr, "seed": sd, "best_epoch": h["best_epoch"], "epochs_run": h["epochs_run"], "best_val": h["best_val_loss_std_units"], "seconds": h["seconds"], "peak_gpu_mb": h["peak_gpu_mem_mb"],
                             "final_train": h["history"]["train_loss"][-1], "gap_val_minus_train_at_best": h["history"]["val_loss"][h["best_epoch"]] - h["history"]["train_loss"][h["best_epoch"]]})
            ax.set_title(f"{kind.upper()} test year {yr}", fontsize=8); ax.set_xlabel("epoch"); ax.set_ylabel("masked MSE (std units)"); ax.set_yscale("log"); ax.legend(fontsize=5, ncol=2)
        fig.suptitle(f"{kind.upper()}: validation (solid) / train (dotted) loss for every fold and seed; dashed = best epoch (early-stopping choice)", fontsize=9)
        _sf(f"training_curves_all_{kind}"); plt.show()
    if rows:
        T = pd.DataFrame(rows); T.to_csv(OUT_DIR / "tables" / f"{PLOT_TAG}training_summary.csv", index=False)
        fig, axs = plt.subplots(1, 4, figsize=(18, 3.6))
        for kind, mk in zip(MODELS, ("o", "s")):
            g = T[T.kind == kind]
            for ax, col, lab in zip(axs, ("best_val", "best_epoch", "seconds", "gap_val_minus_train_at_best"), ("best validation loss", "best epoch", "training seconds", "val - train loss at best epoch")):
                ax.scatter(g.fold, g[col], marker=mk, label=kind.upper(), s=18); ax.set_xlabel("test year"); ax.set_ylabel(lab); ax.set_title(lab, fontsize=8); ax.legend(fontsize=7)
        _sf("training_summary"); plt.show(); display(T.round(4))

# ---- actual vs forecast surfaces on the worst / typical / calmest day
def fig_surfaces():
    dm = DAILY["persistence"]; ok = np.flatnonzero(np.isfinite(dm))
    picks = {"worst day (persistence)": ok[np.argmax(dm[ok])], "typical day (median persistence error)": ok[np.argsort(dm[ok])[len(ok) // 2]], "calmest day (persistence)": ok[np.argmin(dm[ok])]}
    offs = np.cumsum([0] + [len(f.test) for f in folds])
    cols = ["persistence", BEST_BASE] + ([BEST_NEURAL] if BEST_NEURAL else [])
    fig, axs = plt.subplots(len(picks), len(cols) + 2, figsize=(3.3 * (len(cols) + 2), 3.4 * len(picks)), squeeze=False)
    for r, (label, gidx) in enumerate(picks.items()):
        fi = int(np.searchsorted(offs, gidx, side="right") - 1); f = folds[fi]; j = gidx - offs[fi]; row = f.test[j]
        tgt = p.targets[row]; act = np.where(_TST[row] == _OBS, _TY[row], np.nan) * 100.0
        lo, hi = np.nanmin(act), np.nanmax(act); date = str(_dt([p.dates[tgt]])[0].date())
        heat(axs[r][0], act.reshape(NT, NK), f"{label}\nactual (observed cells) {date}", "viridis", lo, hi)
        for c, m in enumerate(cols):
            pr = preds_all[f.test_year][m][j] * 100.0; heat(axs[r][c + 1], np.where(np.isfinite(act), pr, np.nan).reshape(NT, NK), f"{m}: forecast", "viridis", lo, hi)
        errm = cols[-1]; er = (preds_all[f.test_year][errm][j] * 100.0 - act).reshape(NT, NK); em = np.nanmax(np.abs(er)) or 1.0
        heat(axs[r][-1], er, f"{errm}: forecast - actual (vol pts)", "RdBu_r", -em, em)
        PLOT_DATA[f"surface_day{r}__actual"] = act.reshape(NT, NK); PLOT_DATA[f"surface_day{r}__error__{errm}"] = er
        for m in cols: PLOT_DATA[f"surface_day{r}__forecast__{m}"] = preds_all[f.test_year][m][j].reshape(NT, NK) * 100.0
    _sf("surface_actual_vs_forecast_heatmaps"); plt.show()

for nm, fn in (("error_heatmaps", fig_error_heatmaps), ("skill_heatmaps", fig_skill_heatmaps), ("level_heatmaps", fig_level_heatmaps), ("daily_series", lambda: (build_daily(), fig_daily())),
               ("breakdown_curves", fig_breakdowns), ("skill_and_seeds", fig_skill_and_seeds), ("training_curves", fig_training), ("surface_days", fig_surfaces)):
    guard(nm, fn)
(OUT_DIR / "tables").mkdir(exist_ok=True)
np.savez_compressed(OUT_DIR / "tables" / f"{PLOT_TAG}plot_data.npz", **{k: np.asarray(v) for k, v in PLOT_DATA.items()})
print(f"plots now in folder: {sorted(x.stem for x in (OUT_DIR / 'plots').glob('*.png'))}")
print("extra-plot errors:", EXTRA_PLOT_ERRORS or "none")
'''
def pool_brk(df):
    g = df.groupby(["group", "model"]); n = g.n_cells.sum()
    out = pd.DataFrame({"rmse": np.sqrt((df.assign(w=df.rmse ** 2 * df.n_cells).groupby(["group", "model"]).w.sum()) / n.replace(0, np.nan)), "n_cells": n})
    return out.reset_index()
E2_PLOT_ERRORS = {}
for PLOT_TAG, p, preds_all, per_fold, pooled, brk, HIST_DIR in (
        ("e1_nearest_", p1, all1, per_fold1, pooled1, brk1, OUT_DIR / "e1_nearest" / "histories"),
        ("e2_fixed_", p2, all2, per_fold2, pooled2, brk2, OUT_DIR / "e2_fixed" / "histories")):
    print(f"===== extra plots for target {PLOT_TAG.rstrip('_')} =====")
    _pr = pooled[pooled.cells == "primary"].sort_values("rmse")
    BEST_NEURAL = next((m for m in _pr.model if m.endswith("_ens")), None)
    BEST_BASE = next(m for m in _pr.model if m in ("rolling_mean", "ridge", "pca_ridge"))
    exec(EXTRA_SRC, globals())
    E2_PLOT_ERRORS[PLOT_TAG] = dict(EXTRA_PLOT_ERRORS)
print("extra-plot errors per target:", E2_PLOT_ERRORS or "none")


## 12. Results and interpretation
Generated from the tables above, so it cannot claim more than was computed.

In [ ]:
import re
c1, c2 = prim(pooled1).set_index("model"), prim(pooled2).set_index("model")
cm = pooled_c.set_index(["model", "target"])
lines = [f"Data: {DATA_LABEL}; {audit['n_surface_dates']} surface dates {audit['first_date']} -> {audit['last_date']}; {len(folds)} walk-forward test years {[f.test_year for f in folds]}.",
         f"Neural models executed on {DEVICE['name']}: {len(cfg.train.seeds)} seeds x {len(cfg.train.models)} architectures x {len(folds)} folds x 2 targets.",
         f"Eligible test cells: E-1 {int(c1.loc['persistence','n_cells']):,}; E-2 {int(c2.loc['persistence','n_cells']):,}; common sample {int(cm.loc[('persistence','fixed_expiry'),'n_cells']):,}.",
         f"Persistence RMSE on the common sample: nearest-expiry label {cm.loc[('persistence','nearest_expiry'),'rmse']:.3f} pts vs fixed-expiry label {cm.loc[('persistence','fixed_expiry'),'rmse']:.3f} pts."]
for m in [x for x in c2.index if x != "persistence" and not re.search(r"_s\d+$", x)]:
    a = cm.loc[(m, "fixed_expiry"), "mse_skill_vs_persistence"]; b = cm.loc[(m, "nearest_expiry"), "mse_skill_vs_persistence"]
    row = paired2[paired2.model == m].iloc[0]
    lines.append(f"- {m}: MSE skill vs persistence on the common sample: fixed-expiry {a:+.3f} vs nearest-expiry {b:+.3f}; under E-2 better than persistence in {row.folds_better_than_persistence} test years, on {row.share_of_days_better:.1%} of days (n={int(row.n_days)}).")
beaten2 = [m for m in c2.index if m != "persistence" and not re.search(r"_s\d+$", m) and c2.loc[m, "mse_skill_vs_persistence"] > 0]
lines.append("Models with positive pooled MSE skill vs persistence under the fixed-expiry label (own eligible cells): " + (", ".join(beaten2) if beaten2 else "none") + ".")
lines += ["Not concluded: statistical significance (unsupported); that a model 'improves' on persistence unless the per-year table shows it consistently; generalisation beyond this dataset; any trading or hedging value.",
          "Caveats: the two labels have different eligible samples (use the common-sample rows to compare); E-2 censors cells whose expiry vanishes and excludes interpolated-in-maturity cells; closing-snapshot mids; American-vs-European mismatch."]
INTERPRETATION_TEXT = "\n".join(lines); print(INTERPRETATION_TEXT)


## 13. Save outputs
One run folder with a SHA-256 manifest, re-opened and re-hashed. The raw dataset is never copied.

In [ ]:
import platform
for name, df in (("coverage", cov), ("pooled_e1_nearest", pooled1), ("pooled_e2_fixed", pooled2), ("per_fold_e1", per_fold1), ("per_fold_e2", per_fold2),
                 ("paired_e2_vs_persistence", paired2), ("pooled_common", pooled_c), ("per_fold_common", per_c), ("switch_breakdown", sb), ("tenor_common", tenor_c),
                 ("e1_reproduction", repro.reset_index()), ("folds", fold_table), ("neural_runs", HIST)):
    df.to_csv(OUT_DIR / "tables" / f"{name}.csv", index=False)
runner.save_panel(panel, OUT_DIR / "surfaces" / "surface_panel.npz")
runner.export_forecasts(p1, folds, all1, OUT_DIR / "e1_nearest")   # forecasts/<model>.npz for the viewer API
runner.export_forecasts(p2, folds, all2, OUT_DIR / "e2_fixed")     # actual shown there is the surface at the target date, not the E-2 label
for tag, p, preds in (("e1", p1, all1), ("e2", p2, all2)):
    for f in folds:
        ty, tst = target_arrays(p)
        np.savez_compressed(OUT_DIR / "predictions" / f"{tag}_eval_fold{f.test_year}.npz", origin_dates=p.dates[p.origins[f.test]], target_dates=p.dates[p.targets[f.test]],
            actual=ty[f.test], target_status=tst[f.test], primary_mask=Mx.eval_mask(p, f.test, "primary"), common_mask=compare.common_mask(p1, p2, f.test), cell_tenor=p.cell_tenor, cell_k=p.cell_k,
            models=np.array(sorted(preds[f.test_year])), **{f"pred__{m}": v for m, v in preds[f.test_year].items()})
meta = {"run_id": RUN_ID, "experiment": "H4 (variant of E-2)", "variant": FEATURE_DEFS["variant"], "near_untrained_runs": NEAR_UNTRAINED, "decision_rule": {m: {"verdict": r["verdict"], **{l: {k: r[l][k] for k in ("a", "b", "c", "d", "label_pass")} for l in ("nearest_expiry", "fixed_expiry")}} for m, r in RULE.items()}, "data_label": DATA_LABEL, "kaggle_dataset": KAGGLE_DATASET, "dataset_fingerprint": FINGERPRINT,
        "config": cfg.to_dict(), "config_sha256": cfg.sha256(), "feature_definitions": FEATURE_DEFS, "split_boundaries": fold_table.to_dict("records"), "leakage_audit": LEAK,
        "ledger_prefilter": built["ledger_prefilter"], "ledger_quotes": built["ledger_quotes"], "audit": audit, "device": DEVICE, "neural_seconds": NEURAL_SECONDS, "extra_plot_errors": E2_PLOT_ERRORS,
        "environment": runner.environment_metadata(), "colab": {"platform": platform.platform()}}
(OUT_DIR / "metadata").mkdir(exist_ok=True)
(OUT_DIR / "metadata" / "run_metadata.json").write_text(json.dumps(meta, indent=1, default=str))
def md(df, **kw): return "```\n" + df.to_string(**kw) + "\n```"
report = ["# H4 run report", f"**{DATA_LABEL}** | run `{RUN_ID}`", "", "## Interpretation (generated)", INTERPRETATION_TEXT, "", "## Eligible-sample coverage", md(cov, index=False), "",
          "## Pooled, E-2 fixed expiry (primary cells)", md(prim(pooled2).round(4), index=False), "", "## Pooled, E-1 nearest expiry re-run", md(prim(pooled1).round(4), index=False), "",
          "## Decision rule", md(RULE_TABLE[["model", "label", "a_pass", "b_pass", "c_pass", "d_pass", "label_pass", "verdict_both_labels"]], index=False), "", "## Common sample", md(tab), "", "## Gates", "- inferential validity: UNSUPPORTED", "- trading / hedging validity: NOT EVALUATED"]
(OUT_DIR / "report.md").write_text("\n".join(report))
man = runner.write_manifest(OUT_DIR, {"run_id": RUN_ID, "config_sha256": cfg.sha256(), "dataset_combined_sha256": FINGERPRINT["combined_sha256"]})
problems = runner.verify_manifest(OUT_DIR); assert problems == [], problems
print(f"{len(man['files'])} files written, re-opened and re-hashed OK under {OUT_DIR}")
for rel in ("report.md", "manifest.json", "tables/pooled_common.csv", "tables/coverage.csv", "tables/pooled_e2_fixed.csv", "surfaces/surface_panel.npz", "metadata/run_metadata.json", "plots/e2_tenor_and_year.png"):
    fp = OUT_DIR / rel; print(("OK  " if fp.exists() else "MISSING "), fp, f"({fp.stat().st_size:,} bytes)" if fp.exists() else "")
print("checkpoints:", len(list(OUT_DIR.rglob("*.pt"))), "| prediction files:", len(list((OUT_DIR / "predictions").glob("*.npz"))), "| plots:", len(list((OUT_DIR / "plots").glob("*.png"))))
print("re-opened a predictions file:", np.load(OUT_DIR / "predictions" / f"e2_eval_fold{folds[0].test_year}.npz")["actual"].shape)
print("RESULT SUMMARY -> H4 | config", cfg.sha256()[:8], "| near-untrained runs:", NEAR_UNTRAINED, "of", len(HIST), "| verdicts:", {m: r["verdict"] for m, r in RULE.items()})


## 14. Reproducibility
1. **Open** this notebook in Google Colab (File -> Open notebook -> GitHub/Drive/Upload).
2. **GPU:** Runtime -> Change runtime type -> GPU (T4 or better). Section 3 prints the device; neural cells refuse to run without one.
3. **Data access:** add `KAGGLE_USERNAME` and `KAGGLE_KEY` (or `KAGGLE_API_TOKEN`) as Colab Secrets with *Notebook access* on. Nothing else is needed: no manual download. (`RUN_MODE='synthetic_plumbing'` needs no credentials but is synthetic.)
4. **Run all** cells in order (Runtime -> Run all). Settings live only in Section 2.
5. **Resume:** finished (fold, model, seed) checkpoints, predictions and histories are reused when the config and dataset bytes are unchanged (the run id encodes both). Set `USE_DRIVE = True` to keep them across runtime resets.
6. **Results:** `report.md`, `tables/`, `plots/`, `predictions/`, `checkpoints/`, `surfaces/` and `manifest.json` (SHA-256 of every file) under the run folder printed in Section 5.
7. **Not claimed:** statistical significance, generalisation beyond SPY / this vendor / this period, or any trading value.
